<a href="https://colab.research.google.com/github/peraltanadine/TPI_IA/blob/main/TPI_Inteligencia_Artificial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TPI — Decision Tree (C4.5) vs Random Forest

**Versión 12**

**IA · 5º ISI · UTN FRRe · Equipo Buy(Milk)**

Vamos a implementar un **árbol de decisión desde cero** (estilo C4.5 de Quinlan) y compararlo con un **Random Forest** de Scikit-learn: mismo problema de clasificación, mismas métricas, distintas “filosofías” (un árbol interpretable vs un ensamble más robusto).

Este notebook es la **guía de construcción**. Cada bloque dice *qué es*, *por qué lo hacemos* y *qué codear*. El detalle fino de Trello está en `buy-milk-/docs/cards/`.

**Entrega:** 11/11/2026 · Colab + informe LNCS ≤ 12 carillas + Trello activo.

Vamos a implementar un **árbol de decisión desde cero** (estilo C4.5 de Quinlan) y compararlo con un **Random Forest** de Scikit-learn: mismo problema de clasificación, mismas métricas, distintas “filosofías” (un árbol interpretable vs un ensamble más robusto).

Este notebook es la **guía de construcción**. Cada bloque dice *qué es*, *por qué lo hacemos* y *qué codear*. El detalle fino de Trello está en `buy-milk-/docs/cards/`.

**Entrega:** 11/11/2026 · Colab + informe LNCS ≤ 12 carillas + Trello activo.


---
## Quién hace qué

| Rol | Persona | De qué se ocupa |
|---|---|---|
| **Implementador DT** | Celeste Martin Rodich | Árbol base: nodo, entropía, GainRatio, splits categóricos, `fit` / `predict` |
| **Ingeniero de datos** | Matías Emmanuel Machado Benítez | Carga, EDA, limpieza open–closed y, después, splits **continuos** |
| **Especialista C4.5** | Camila Aylen Zeniquel Martinelli | Completar C4.5: parada, faltantes, poda, API tipo sklearn, tests |
| **Analista experimental** | Nadine Andrea Peralta Ruiz | Random Forest, métricas, validación cruzada, experimentos E1… |
| **Integrador** | Mauricio Nicolás Castro | Andamiaje, Colab/widgets, UI, figuras, bitácora, cierre del informe |

**Por qué así:** el DT completo es mucho para una sola persona, pero si dos tocan `fit` a la vez se pisan. Entonces el Implementador DT deja un árbol categórico que **funciona**; el Especialista C4.5 lo **extiende**; el Ingeniero de datos aporta umbrales continuos sin reescribir el orquestador. El Analista experimental concentra la evaluación para que DT y RF no se midan con reglas distintas. El Integrador sostiene entorno, UI y entrega.

**Orden mental del proyecto**

1. Preparar datos y entorno  
2. Árbol categórico mínimo que predice  
3. Sumar continuos / faltantes / poda  
4. Medir y comparar con RF  
5. UI + informe + entrega

---
# 0. Andamiaje (antes de codear en serio)

**Slot: 5º** · Cards Trello 1–6

**Qué es:** dejar el proyecto usable (repo, Trello, LaTeX, preguntar el dataset a la cátedra).

**Por qué:** sin dataset confirmado no frenamos: usamos un **respaldo** (p.ej. Breast Cancer de sklearn) y después cambiamos una línea. El Trello lo revisa la cátedra: tiene que verse actividad real, no todo el último día.

**Hacer:** carpetas del repo, board con listas/labels, asignar nombres a los slots, verificar que compile `informe/`, consultar dataset oficial.

---
# 1. Datos y entorno

Sin datos limpios y un runtime reproducible, el resto del TP es ruido.

### 1.1 Setup

**Slot: 5º** · Card 7

**Hacer:** imports (`numpy`, `pandas`, `sklearn`, `matplotlib`, `ipywidgets`), imprimir versiones, definir `SEED` y constantes (`RUTA_DATOS`, nombre del `TARGET`).

Esta celda se ejecuta primero y prepara el entorno de todo el notebook. Importa las librerías del proyecto (NumPy, pandas, scikit-learn, matplotlib, ipywidgets), imprime sus versiones para documentar el entorno de ejecución y define las constantes globales: la semilla SEED, la ruta del dataset RUTA_DATOS, la columna clase TARGET_COL y las columnas de atributos FEATURE_COLS.

Fijar la semilla hace que cada ejecución produzca exactamente los mismos resultados (particiones train/test, árboles del Random Forest, folds de validación cruzada). Sin eso, las diferencias entre DT y RF podrían deberse al azar y no a los modelos, y los resultados del informe no serían verificables.

**Reglas del equipo**

* Todos los imports van en esta celda. No se importan librerías en otras celdas.
* Todo lo que use azar recibe random_state=SEED en forma explícita (train_test_split, RandomForestClassifier, StratifiedKFold, RepeatedStratifiedKFold, etc.). No se confía solo en la semilla global.
* Ningún valor configurable se escribe "a mano" en otras celdas. El target, la ruta y la semilla se leen de las constantes de esta celda.
* No se silencian todos los warnings: solo los de deprecación. Los demás pueden indicar errores reales.
* Antes de subir cambios, se verifica con Runtime → Restart and run all: dos ejecuciones seguidas deben dar las mismas métricas.



In [1]:
# ═══ 1.1 Setup: librerías, versiones, semilla y constantes ═══
# Esta celda se corre PRIMERO. Define todo lo que el resto del notebook asume.

import sys, io, random, warnings

import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets
import seaborn as sns
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold

# ── Reproducibilidad ─────────────────────────────────────────────
# SEED fija el azar: misma semilla → mismos splits, mismos bosques, mismas métricas.
# REGLA DEL EQUIPO: todo lo que tenga azar recibe random_state=SEED explícito. ←-----------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Protocolo único de medición (card 33). 5 folds × 5 repeats = 25 puntuaciones por métrica.
# Se define una sola vez: si cada celda arma el suyo, DT y RF dejan de ser comparables.
# RepeatedStratifiedKFold estratifica cada fold; sin eso, una clase puede faltar en algún fold.
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

# ── Constantes del proyecto ──────────────────────────────────────
RUTA_DATOS   = "data/dataset.csv"  # None → usa el dataset de respaldo (Breast Cancer)
TARGET_COL   = "target"            # columna clase. "target" es el nombre en el respaldo
FEATURE_COLS = None                # None → todas las columnas menos el target
DIR_FIGURAS = Path("figuras")      # acá se guardan figuras y tablas del informe

# ── Warnings: silenciamos solo ruido de versiones, no errores reales ──
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── Versiones (van al informe) ───────────────────────────────────
print(f"Python      {sys.version.split()[0]}")
print(f"NumPy       {np.__version__}")
print(f"Pandas      {pd.__version__}")
print(f"sklearn     {sklearn.__version__}")
print(f"matplotlib  {matplotlib.__version__}")
print(f"ipywidgets  {widgets.__version__}")
print(f"SEED        {SEED}")
print(f"cv          {cv.get_n_splits()} puntuaciones  repeats={cv.n_repeats}  seed={cv.random_state}")
print("✓ Setup listo")

Python      3.12.10
NumPy       2.2.2
Pandas      3.0.5
sklearn     1.9.0
matplotlib  3.11.1
ipywidgets  8.1.9
SEED        42
cv          25 puntuaciones  repeats=5  seed=42
✓ Setup listo


### 1.2 Cargar el dataset

**Slot: 2º** · Cards 8 y 12

**Por qué:** el DT y el RF no deberían saber *de dónde* salieron los datos. Si mañana cambia el CSV de la cátedra, solo tocamos el loader.

**Hacer:** si existe el CSV → leerlo; si no → dataset de respaldo. Dejar claro qué columna es la **clase** (target) y cuáles son los **atributos** (features).

Si está `data/heart+disease`, esta celda junta Cleveland, Hungría, Suiza y Long Beach. La clase 0 queda como ausencia y 1–4 como presencia. El `?` queda faltante, sin imputar. Si esa carpeta no está, lee el CSV de RUTA_DATOS y, si tampoco está, usa Breast Cancer.

Separar la carga del resto del notebook permite que el árbol de decisión y el Random Forest no dependan del origen de los datos. Cuando la cátedra confirme el dataset oficial, alcanza con modificar RUTA_DATOS (y, si corresponde, TARGET_COL y FEATURE_COLS) en la celda de setup, sin cambios en el código de los modelos.

La función valida que la columna clase y las columnas de atributos existan y, si no, informa el error indicando las columnas disponibles. Al finalizar imprime un resumen: dimensiones, tipos de datos y distribución de clases.

**Resultado viejo, solo del respaldo Breast Cancer**

* 569 instancias, 30 atributos numéricos continuos (float64) y 1 columna clase (object).
* Distribución de clases: 357 benign (62,7 %) y 212 malignant (37,3 %). Hay un desbalance moderado, por lo que la accuracy sola no alcanza para evaluar los modelos: se deben reportar también precision, recall y F1, y usar particiones estratificadas.
* Todos los atributos son continuos. Con el dataset de respaldo, el árbol requiere la implementación de umbrales para atributos continuos (tarea 21) para poder entrenarse.

**Reglas del equipo**

* El dataset se carga únicamente con cargar_datos(). Ninguna otra celda lee archivos ni llama a load_breast_cancer.
* El resto del notebook trabaja solo con df, TARGET_COL y FEATURE_COLS. Nunca se escribe el nombre de la columna clase ni de los atributos "a mano".
* Para cambiar de dataset se modifican las constantes de la celda de setup, no esta celda.
* Si el dataset oficial trae columnas que no son atributos (identificadores, fechas de registro, etc.), se excluyen definiendo FEATURE_COLS en el setup, no borrándolas en celdas intermedias.
* Cualquier transformación de los datos (limpieza, imputación, discretización) se hace en la etapa de preprocesamiento (1.4), no dentro del loader.


In [2]:
# ═══ 1.2 Carga del dataset ═══
# Si está data/heart+disease, arma las cuatro sedes acá. Si no, lee el CSV.
# Si tampoco está el CSV, usa Breast Cancer. El resto del notebook ve df.

from sklearn.datasets import load_breast_cancer


def _leer_csv(ruta):
    """Lee un CSV detectando el separador (',' o ';') y probando encodings comunes."""
    for encoding in ("utf-8", "latin-1"):
        try:
            return pd.read_csv(ruta, sep=None, engine="python", encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo leer {ruta} con utf-8 ni latin-1.")


def _cargar_respaldo():
    """Breast Cancer de sklearn con la clase como texto (malignant/benign)."""
    datos = load_breast_cancer(as_frame=True)
    df = datos.frame.copy()
    df[TARGET_COL] = df["target"].map(dict(enumerate(datos.target_names)))
    if TARGET_COL != "target":
        df = df.drop(columns="target")
    return df


def _cargar_corazon():
    """Cuatro sedes UCI. 0 es ausencia; 1–4 es presencia. ? queda faltante."""
    carpeta = Path("data/heart+disease")
    columnas = [
        "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
        "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num",
    ]
    archivos = [
        "processed.cleveland.data",
        "processed.hungarian.data",
        "processed.switzerland.data",
        "processed.va.data",
    ]
    if not all((carpeta / nombre).exists() for nombre in archivos):
        return None
    partes = []
    for nombre in archivos:
        tabla = pd.read_csv(
            carpeta / nombre, header=None, names=columnas, na_values="?", sep=",",
        )
        if tabla.shape[1] != len(columnas):
            raise ValueError(f"{nombre} no tiene {len(columnas)} columnas")
        print(f"  {nombre}: {len(tabla)} filas, {int(tabla.isna().sum().sum())} faltantes")
        partes.append(tabla)
    datos = pd.concat(partes, ignore_index=True)
    if datos["num"].isna().any():
        raise ValueError("Hay filas sin diagnóstico.")
    datos["target"] = datos["num"].map(lambda v: "ausencia" if int(v) == 0 else "presencia")
    datos = datos.drop(columns="num")
    chol0 = int((datos["chol"] == 0).sum())
    print(f"  Cuatro sedes: {len(datos)} filas. Colesterol 0, se deja: {chol0}.")
    print("  Clase: 0 → ausencia, 1–4 → presencia. Sin imputar.")
    Path("data").mkdir(exist_ok=True)
    datos.to_csv("data/dataset.csv", index=False)
    return datos


def _elegir_clase(columnas):
    """La columna configurada, si está. Si no, la última del archivo."""
    nombres = list(columnas)
    if TARGET_COL in nombres:
        return TARGET_COL
    return nombres[-1]


def cargar_datos(ruta=RUTA_DATOS):
    """Devuelve (df, nombre_dataset). La clase sale del archivo."""
    global TARGET_COL
    corazon = _cargar_corazon()
    if corazon is not None:
        df = corazon
        nombre = "heart disease (Cleveland, Hungría, Suiza, Long Beach)"
    elif ruta is not None and Path(ruta).exists():
        df = _leer_csv(ruta)
        df.columns = df.columns.str.strip()
        nombre = Path(ruta).stem
    else:
        if ruta is not None:
            print(f"⚠ No se encontró '{ruta}'. Se usa el dataset de respaldo.")
        df = _cargar_respaldo()
        nombre = "breast_cancer (respaldo sklearn)"

    clase = _elegir_clase(df.columns)
    if clase != TARGET_COL:
        print(f"No está '{TARGET_COL}'. La clase pasa a ser '{clase}'.")
        TARGET_COL = clase

    features = FEATURE_COLS or [c for c in df.columns if c != TARGET_COL]
    faltantes = set(features) - set(df.columns)
    if faltantes:
        raise KeyError(f"Columnas de FEATURE_COLS que no existen: {faltantes}")

    return df[features + [TARGET_COL]], nombre


# ── Ejecución + resumen ──────────────────────────────────────────
df, NOMBRE_DATASET = cargar_datos()

print(f"Dataset : {NOMBRE_DATASET}")
print(f"Shape   : {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Target  : '{TARGET_COL}'\n")
print("Tipos de datos:")
print(df.dtypes.value_counts().to_string(), "\n")
print("Conteo de clases:")
print(df[TARGET_COL].value_counts().to_frame("n").assign(
    pct=lambda t: (t["n"] / t["n"].sum() * 100).round(1)).to_string())


  processed.cleveland.data: 303 filas, 6 faltantes
  processed.hungarian.data: 294 filas, 782 faltantes
  processed.switzerland.data: 123 filas, 273 faltantes
  processed.va.data: 200 filas, 698 faltantes
  Cuatro sedes: 920 filas. Colesterol 0, se deja: 172.
  Clase: 0 → ausencia, 1–4 → presencia. Sin imputar.
Dataset : heart disease (Cleveland, Hungría, Suiza, Long Beach)
Shape   : 920 filas × 14 columnas
Target  : 'target'

Tipos de datos:
float64    13
str         1 

Conteo de clases:
             n   pct
target              
presencia  509  55.3
ausencia   411  44.7


### 1.4 Limpieza open–closed

**Slot: 2º**

**Qué es:** un preprocess que **despacha por tipo de atributo**:
- *discreto / categórico* → una función de limpieza
- *continuo / numérico* → otra
- el orquestador solo mira el tipo y llama a la función correcta

**Por qué:** “open–closed” = abierto a sumar un tipo nuevo, cerrado a reescribir todo el pipeline. Primero hacemos andar lo discreto; después enchufamos continuos sin tocar el árbol.

**Ojo:** limpiar nulos en el DataFrame **no es lo mismo** que el manejo fraccional de faltantes dentro de C4.5 (eso viene después). Decidan qué hace cada capa y lo anotan.

**Hacer:** `infer_attribute_types` → `clean_discrete` / `clean_continuous` → `preprocess` que las use.

In [3]:
# ═══ 1.4 Tipos, sin imputar ═══
# Texto y bool: una rama por valor. Un entero con pocos valores también:
# es un código, no una medida. Los nulos se quedan.

# Hasta 10 enteros distintos. En Heart Disease los códigos llegan a 4 y la edad a 50.
_TOPE_CODIGO = 10


def _es_codigo(serie):
    """Entero con pocos valores distintos."""
    if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
        return False
    numeros = pd.to_numeric(serie, errors="coerce").dropna()
    if numeros.empty or not (numeros % 1 == 0).all():
        return False
    return int(numeros.nunique()) <= _TOPE_CODIGO


def infer_attribute_types(datos, target):
    """Separa continuas y categóricas mirando los valores, no el nombre."""
    continuas, categoricas, otras = [], [], []
    for col in datos.columns:
        if col == target:
            continue
        serie = datos[col]
        if (
            pd.api.types.is_bool_dtype(serie)
            or _es_codigo(serie)
            or not pd.api.types.is_numeric_dtype(serie)
        ):
            categoricas.append(col)
        else:
            continuas.append(col)
    return {"continua": continuas, "categorica": categoricas, "otra": otras}


def clean_continuous(serie):
    """Pasa a número. No rellena faltantes."""
    return pd.to_numeric(serie, errors="coerce")


def clean_discrete(serie):
    """Código, texto o bool como categoría. El faltante sigue vacío."""
    def etiqueta(valor):
        if isinstance(valor, bool):
            return "true" if valor else "false"
        if pd.isna(valor):
            return pd.NA
        try:
            numero = float(valor)
        except (TypeError, ValueError):
            return str(valor)
        if numero.is_integer():
            return str(int(numero))
        return str(valor)
    return serie.map(etiqueta).astype("category")


DISPATCH = {
    "continua": clean_continuous,
    "categorica": clean_discrete,
    "otra": clean_discrete,
}


def preprocess(datos, target=None):
    """Aplica el despacho por tipo. No imputa."""
    target = TARGET_COL if target is None else target
    salida = datos.copy()
    tipos = infer_attribute_types(salida, target)
    for tipo, columnas in tipos.items():
        for col in columnas:
            salida[col] = DISPATCH[tipo](salida[col])
    return salida


df = preprocess(df)
_tipos = infer_attribute_types(df, TARGET_COL)
print(
    f"Continuas: {_tipos['continua']}\n"
    f"Categóricas: {_tipos['categorica'] or _tipos['otra']}\n"
    f"Faltantes que se dejan: {int(df.isna().sum().sum())}"
)


Continuas: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categóricas: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
Faltantes que se dejan: 1759


### 1.3 EDA (mirar antes de modelar)

**Slot: 2º** · Card 9

**Hacer:** resúmenes + 2–3 figuras con nombre estable (después van al informe).

Esta celda explora el dataset antes de entrenar los modelos, para detectar características que condicionan el diseño de la solución y la forma de evaluarla. Separa automáticamente los atributos numéricos de los categóricos, por lo que funciona con cualquier dataset cargado por cargar_datos(). Genera la Tabla 1 (resumen por atributo: tipo, faltantes, valores únicos, media, desvío, mínimo y máximo) y las figuras de la Fig. 2 (distribución de clases, distribución de cada atributo por clase y matriz de correlación), que se guardan en DIR_FIGURAS con nombres fijos para incorporarlas al informe.

**Reglas del equipo**

* El EDA no modifica df. Cualquier limpieza o transformación se hace en la etapa de preprocesamiento (1.4).
* Las figuras y tablas se guardan siempre en DIR_FIGURAS con nombres fijos. Si se cambia una figura, se sobrescribe el archivo, no se crea uno nuevo con otro nombre.
* Las decisiones que surjan del EDA (por ejemplo, excluir un atributo tipo identificador) se aplican desde las constantes del setup (FEATURE_COLS), no borrando columnas en celdas intermedias.
* La línea base de la clase mayoritaria (62,7 % con el respaldo) se reporta junto a los resultados de DT y RF, para que las métricas tengan una referencia.
* Si se cambia de dataset, se vuelve a correr el EDA y se actualiza este texto antes de seguir modelando.

In [4]:
# ═══ 1.3 EDA: exploración antes de modelar ═══
# Produce la Tabla 1 (resumen del dataset) y las figuras de la Fig. 2 del informe.
# Funciona con cualquier dataset: separa atributos numéricos y categóricos automáticamente.

DIR_FIGURAS.mkdir(exist_ok=True)

features  = [c for c in df.columns if c != TARGET_COL]
num_cols  = df[features].select_dtypes(include="number").columns.tolist()
cat_cols  = [c for c in features if c not in num_cols]

print(f"Dataset: {NOMBRE_DATASET} · {df.shape[0]} filas · "
      f"{len(num_cols)} atributos numéricos · {len(cat_cols)} categóricos\n")

# ── 1. Tabla 1: resumen por columna ──────────────────────────────
tabla1 = pd.DataFrame({
    "tipo":          df[features].dtypes.astype(str),
    "faltantes":     df[features].isna().sum(),
    "% faltantes":   (df[features].isna().mean() * 100).round(2),
    "valores únicos": df[features].nunique(),
})
if num_cols:
    tabla1 = tabla1.join(df[num_cols].describe().T[["mean", "std", "min", "max"]].round(3))

display(tabla1)
tabla1.to_csv(DIR_FIGURAS / "tabla1_resumen_dataset.csv")

# ── 2. Faltantes ─────────────────────────────────────────────────
total_faltantes = int(df.isna().sum().sum())
if total_faltantes == 0:
    print("✓ Sin valores faltantes.")
else:
    print(f"⚠ {total_faltantes} valores faltantes en "
          f"{(df.isna().sum() > 0).sum()} columnas → aplica manejo de faltantes de C4.5.")

# ── 3. Distribución de clases ────────────────────────────────────
conteo = df[TARGET_COL].value_counts()
ratio  = conteo.max() / conteo.min()
print(f"\nClases: { {str(k): int(v) for k, v in conteo.items()} } · ratio mayoritaria/minoritaria = {ratio:.2f}")
if ratio > 1.5:
    print("⚠ Desbalance: reportar precision/recall/F1 macro y usar particiones estratificadas.")

fig, ax = plt.subplots(figsize=(5, 3.5))
conteo.plot.bar(ax=ax, color="steelblue", edgecolor="black")
for i, v in enumerate(conteo):
    ax.text(i, v, f"{v} ({v / conteo.sum():.1%})", ha="center", va="bottom")
ax.set(title="Distribución de clases", xlabel=TARGET_COL, ylabel="Instancias")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
fig.savefig(DIR_FIGURAS / "fig2a_distribucion_clases.png", dpi=300)
plt.show()

# ── 4. Distribución de cada atributo, separada por clase ─────────
if num_cols:
    n_cols_grid = 5
    n_filas = -(-len(num_cols) // n_cols_grid)          # división redondeando hacia arriba
    fig, axes = plt.subplots(n_filas, n_cols_grid, figsize=(3.2 * n_cols_grid, 2.4 * n_filas))
    axes = np.atleast_1d(axes).ravel()
    for ax, col in zip(axes, num_cols):
        for clase in conteo.index:
            ax.hist(df.loc[df[TARGET_COL] == clase, col].dropna(),
                    bins=25, alpha=0.55, label=str(clase))
        ax.set_title(col, fontsize=9)
        ax.tick_params(labelsize=7)
    for ax in axes[len(num_cols):]:
        ax.set_visible(False)
    axes[0].legend(fontsize=8)
    fig.suptitle("Distribución de atributos numéricos por clase", y=1.0)
    fig.tight_layout()
    fig.savefig(DIR_FIGURAS / "fig2b_distribuciones.png", dpi=300, bbox_inches="tight")
    plt.show()

for col in cat_cols:   # categóricos: frecuencia de cada valor (los 15 más comunes)
    print(f"\n{col} ({df[col].nunique()} valores distintos):")
    print(df[col].value_counts().head(15).to_string())

# ── 5. Matriz de correlación ─────────────────────────────────────
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    fig, ax = plt.subplots(figsize=(0.35 * len(num_cols) + 3, 0.35 * len(num_cols) + 2))
    sns.heatmap(corr, cmap="coolwarm", vmin=-1, vmax=1, center=0,
                square=True, annot=False, cbar_kws={"shrink": 0.7}, ax=ax)
    ax.set_title("Matriz de correlación (Pearson)")
    fig.tight_layout()
    fig.savefig(DIR_FIGURAS / "fig2c_correlacion.png", dpi=300)
    plt.show()

    # Pares muy correlacionados (solo triángulo superior, para no repetir)
    superior = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    pares = superior.stack().loc[lambda s: s.abs() >= 0.9].sort_values(key=abs, ascending=False)
    print(f"\nPares con |r| ≥ 0.9: {len(pares)}")
    print(pares.round(3).head(10).to_string())

Dataset: heart disease (Cleveland, Hungría, Suiza, Long Beach) · 920 filas · 5 atributos numéricos · 8 categóricos



,tipo,faltantes,% faltantes,valores únicos,mean,std,min,max
age,float64,0,0.00,50,53.511,9.425,28.0,77.0
sex,category,0,0.00,2,NaN,NaN,NaN,NaN
cp,category,0,0.00,4,NaN,NaN,NaN,NaN
trestbps,float64,59,6.41,61,132.132,19.066,0.0,200.0
chol,float64,30,3.26,217,199.130,110.781,0.0,603.0
fbs,category,90,9.78,2,NaN,NaN,NaN,NaN
restecg,category,2,0.22,3,NaN,NaN,NaN,NaN
thalach,float64,55,5.98,119,137.546,25.926,60.0,202.0
exang,category,55,5.98,2,NaN,NaN,NaN,NaN
oldpeak,float64,62,6.74,53,0.879,1.091,-2.6,6.2


⚠ 1759 valores faltantes en 10 columnas → aplica manejo de faltantes de C4.5.

Clases: {'presencia': 509, 'ausencia': 411} · ratio mayoritaria/minoritaria = 1.24


C:\Users\castr\AppData\Local\Temp\ipykernel_7400\2488446707.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\castr\AppData\Local\Temp\ipykernel_7400\2488446707.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



sex (2 valores distintos):
sex
1    726
0    194

cp (4 valores distintos):
cp
4    496
3    204
2    174
1     46

fbs (2 valores distintos):
fbs
0    692
1    138

restecg (3 valores distintos):
restecg
0    551
2    188
1    179

exang (2 valores distintos):
exang
0    528
1    337

slope (3 valores distintos):
slope
2    345
1    203
3     63

ca (4 valores distintos):
ca
0    181
1     67
2     41
3     20

thal (3 valores distintos):
thal
3    196
7    192
6     46



Pares con |r| ≥ 0.9: 0
Series([], )


C:\Users\castr\AppData\Local\Temp\ipykernel_7400\2488446707.py:85: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 1.5 Probar widgets temprano

**Slot: 5º** · Card 11

Smoke test de `ipywidgets`: un `Dropdown`, un `Button` y un `Output`. No es el panel del sprint 6.

Al correr la celda tiene que imprimirse el fallback, aunque el botón no se dibuje. Si el botón aparece, el click imprime la clase elegida y borra la salida anterior (`clear_output(wait=True)`). Un segundo click muestra el número 2 y no deja las dos líneas juntas.

Anotar en la bitácora de Mauricio, pase lo que pase: entorno (Colab o no), versión de `ipywidgets`, si el botón se ve y si el click imprime.

In [5]:
# ═══ 1.5 Smoke test de ipywidgets (card 11) ═══
# Prueba mínima: Button + Output. El click tiene que imprimir y reemplazar la salida anterior.
# Fallback: el mismo aviso se imprime en la celda. Si Colab no dibuja el widget, Run all sigue.

import ipywidgets as widgets
from IPython.display import display

try:
    import google.colab  # noqa: F401
    _entorno_widgets = "Google Colab"
except ImportError:
    _entorno_widgets = "fuera de Colab"

_selector_prueba = widgets.Dropdown(
    options=["benign", "malignant"],
    value="benign",
    description="Clase:",
)
_boton_prueba = widgets.Button(description="Probar click", button_style="primary")
_salida_prueba = widgets.Output()
_clicks_prueba = {"n": 0}

def _al_click_prueba(_boton):
    _clicks_prueba["n"] += 1
    with _salida_prueba:
        _salida_prueba.clear_output(wait=True)
        print(
            f"ipywidgets responde. Click {_clicks_prueba['n']}. "
            f"Clase elegida: {_selector_prueba.value}. "
            f"Versión {widgets.__version__}. Entorno: {_entorno_widgets}."
        )

_boton_prueba.on_click(_al_click_prueba)

print("Fallback de la celda (card 11): este texto queda aunque el botón no se dibuje.")
print(f"ipywidgets {widgets.__version__}. Entorno: {_entorno_widgets}. Dropdown inicial: {_selector_prueba.value}.")
print("Si ves el botón, hacé click. La línea tiene que aparecer debajo y un segundo click tiene que reemplazarla.")
try:
    display(widgets.VBox([_selector_prueba, _boton_prueba, _salida_prueba]))
except Exception as exc:
    print(f"El widget no se pudo mostrar: {type(exc).__name__}: {exc}")
    print("Si falta la librería: pip install ipywidgets y reiniciar el runtime. Después, volver a correr desde el setup.")

Fallback de la celda (card 11): este texto queda aunque el botón no se dibuje.
ipywidgets 8.1.9. Entorno: fuera de Colab. Dropdown inicial: benign.
Si ves el botón, hacé click. La línea tiene que aparecer debajo y un segundo click tiene que reemplazarla.


---
# 2. El Decision Tree base (C4.5, parte 1)

**Idea en una frase:** partimos los datos preguntando por el atributo que más **ordena** las clases; repetimos en cada rama hasta poder etiquetar una hoja.

Eso es aprendizaje supervisado con un modelo interpretable: cada camino del árbol es una regla *si … entonces clase*.

### 2.1 El `Nodo`

**Slot: 1º** · Card 13

**Qué es:** la pieza del árbol. Puede ser:
- **decisión:** “miro el atributo X” (y un umbral si es continuo)
- **hoja:** “predigo la clase Y” (con la distribución de ejemplos que llegaron acá)

**Por qué:** todo `fit` / `predict` / poda habla el idioma del nodo. Si la estructura sale mal, el resto se complica.

**Hacer:** clase con distribución de clases, atributo, umbral, hijos, `es_hoja`. Acordar: rama izquierda = `<= umbral`.

In [6]:
# Estructura base de cada nodo del árbol. Sirve tanto para nodos internos (con atributo y ramas) como para hojas (con clase asignada).

class Nodo:
    def __init__(self):
        self.atributo = None   # columna usada para el split
        self.hijos    = {}     # {valor_del_atributo: Nodo}
        self.clase    = None   # solo en hojas: clase mayoritaria
        self.es_hoja  = False
        self.conteos  = {}     # distribución de clases en este nodo

    def __repr__(self):
        if self.es_hoja:
            return f"Hoja(clase={self.clase}, conteos={self.conteos})"
        return f"Nodo(atributo='{self.atributo}', ramas={list(self.hijos.keys())})"

# Entropía de Shannon. Recibe cualquier iterable de etiquetas y devuelve un float entre 0 y log₂(k).

def entropy(y):
    """Entropía de Shannon de un array de etiquetas."""
    n = len(y)
    if n == 0:
        return 0.0
    probs = pd.Series(y).value_counts(normalize=True)
    probs = probs[probs > 0]
    return float(-np.sum(probs * np.log2(probs)))


# ── Verificación rápida ───────────────────────────────────────────
assert round(entropy(["a","a","b","b"]), 6) == 1.0,      "debería ser 1.0"
assert entropy(["a","a","a"]) == 0.0,                     "clase pura → 0"
assert round(entropy(["a","b","c"]), 4) == round(np.log2(3), 4), "3 equiprobables"
print("✓ entropy OK")

✓ entropy OK


### 2.2 Entropía, ganancia y Gain Ratio

**Slot: 1º** · Cards 14–16

**Qué es cada cosa (breve):**

| Concepto | En criollo |
|---|---|
| **Entropía** | Qué tan “mezcladas” están las clases. 0 = puro (una sola clase). Alto = un lío. |
| **Ganancia (Gain)** | Cuánto baja la entropía si partimos por un atributo. Más ganancia ⇒ mejor pregunta. |
| **SplitInfo** | Penaliza atributos con *muchos* valores distintos (tipo ID). |
| **Gain Ratio** | `Gain / SplitInfo`. Es el criterio de **C4.5**: elige bien sin enamorarse del atributo con mil categorías. |

**Por qué:** sin esto el árbol no sabe *dónde* cortar. Gain sola favorece IDs; Gain Ratio corrige eso (lo vieron en el ejercicio de cátedra: Edad vs Hipertensión).

**Hacer:** `entropy`, `gain`, `split_info`, `gain_ratio`, y “elegir el mejor atributo”. Cuidado: `0 * log2(0)` tiene que dar 0, no `nan`. Validar con `teoria.ipynb`.

In [7]:
# gain mide cuánto reduce la entropía dividir por un atributo.
# split_info mide cuántas ramas genera ese split (penaliza atributos con muchos valores).

def gain(df, atributo, target):
    """Ganancia de información al dividir df por 'atributo'."""
    n = len(df)
    entropia_padre = entropy(df[target])
    entropia_hijos = sum(
        (len(sub) / n) * entropy(sub[target])
        for sub in (df[df[atributo] == v] for v in df[atributo].unique())
        if len(sub) > 0
    )
    return entropia_padre - entropia_hijos

def split_info(df, atributo):
    """Split information: penaliza atributos con muchas ramas."""
    n = len(df)
    probs = df[atributo].value_counts(normalize=True)
    probs = probs[probs > 0]
    return float(-np.sum(probs * np.log2(probs)))


# ── Verificación rápida ───────────────────────────────────────────
_test = pd.DataFrame({"color": ["rojo","rojo","azul","azul"],
                       "juega": ["si","si","no","no"]})
assert gain(_test, "color", "juega") == 1.0, "split perfecto → gain = H(padre)"
assert split_info(_test, "color") == 1.0,    "dos valores equiprobables → SI = 1"
print("✓ gain y split_info OK")

# gain_ratio normaliza la ganancia por el split_info para no favorecer atributos con muchos valores distintos.
# seleccionar_atributo elige el mejor candidato de la lista disponible.

def gain_ratio(df, atributo, target):
    """Gain ratio = gain / split_info (C4.5)."""
    si = split_info(df, atributo)
    if si == 0:
        return 0.0
    return gain(df, atributo, target) / si


def seleccionar_atributo(df, atributos, target):
    """Devuelve el atributo con mayor gain_ratio."""
    ratios = {a: gain_ratio(df, a, target) for a in atributos}
    return max(ratios, key=ratios.get)

✓ gain y split_info OK


### 2.3 Partir atributos discretos

**Slot: 1º** · Card 17

**Qué es:** si el atributo es categórico (ej. color = rojo/verde/azul), creamos **una rama por valor**.

**Por qué:** es el caso más simple de C4.5 y el primero que tiene que andar. Dejamos un `best_split(tipo)` para que después el continuo se enganche sin reescribir `fit`.

**Hacer:** `best_split_discrete` + diccionario/dispatch `discrete` / `continuous`.

In [8]:
# Divide un DataFrame en un dict de subconjuntos, uno por cada valor único del atributo.
# Solo columnas categóricas (object/category). Las numéricas quedan para la tarea 21.

def split_categorico(df, atributo):
    """Devuelve {valor: subdf} para cada valor único del atributo."""
    return {v: df[df[atributo] == v].reset_index(drop=True)
            for v in df[atributo].unique()}


def _es_categorica(serie):
    """object, category o texto. Pandas 3 guarda strings como StringDtype."""
    if pd.api.types.is_bool_dtype(serie) or pd.api.types.is_numeric_dtype(serie):
        return False
    return (
        serie.dtype in ("object", "category", "string", "str")
        or pd.api.types.is_string_dtype(serie)
    )


def atributos_categoricos(df, target):
    """Filtra columnas categóricas (object/category/texto), excluyendo el target."""
    return [
        c for c in df.columns
        if c != target and (df[c].dtype in ("object", "category") or _es_categorica(df[c]))
    ]

### 2.4 Entrenar y predecir (versión mínima)

**Slot: 1º** · Cards 18 y 20

**Qué es:**
- `fit`: arma el árbol (recursión: elegir atributo → partir → repetir)
- `predict`: baja por las ramas hasta una hoja y devuelve la clase
- `export_text`: imprime el árbol como reglas (para el informe y para debuggear)

**Por qué:** con esto ya tenemos un clasificador propio. Todavía puede ser tosco (sin poda, sin continuos), pero ya se puede medir.

**Hacer:** clase `DecisionTreeC45` con parada mínima (puro o sin atributos). Después el 3º le suma hiperparámetros finos.

In [9]:
class ArbolC45:
    def __init__(self, min_instancias=2, profundidad_max=None):
        self.min_instancias  = min_instancias
        self.profundidad_max = profundidad_max
        self.raiz   = None
        self.target = None

    # ── Entrenamiento ─────────────────────────────────────────────
    def fit(self, df, target):
        self.target = target
        atributos = atributos_categoricos(df, target)
        if not atributos:
            raise ValueError("No hay columnas categóricas para entrenar.")
        self.raiz = self._construir(df, atributos, profundidad=0)
        return self

    def _construir(self, df, atributos, profundidad):
        nodo = Nodo()
        nodo.conteos = df[self.target].value_counts().to_dict()
        clase_mayoritaria = df[self.target].mode()[0]
        nodo.clase = clase_mayoritaria

        # Criterios de parada
        clases_unicas = df[self.target].nunique()
        sin_atributos = not atributos
        pocos_datos   = len(df) < self.min_instancias
        max_prof      = (self.profundidad_max is not None and
                         profundidad >= self.profundidad_max)

        if clases_unicas == 1 or sin_atributos or pocos_datos or max_prof:
            nodo.es_hoja = True
            nodo.clase   = clase_mayoritaria
            return nodo

        mejor = seleccionar_atributo(df, atributos, self.target)
        nodo.atributo = mejor

        atributos_restantes = [a for a in atributos if a != mejor]
        for valor, subdf in split_categorico(df, mejor).items():
            if len(subdf) == 0:
                hijo = Nodo(); hijo.es_hoja = True; hijo.clase = clase_mayoritaria
            else:
                hijo = self._construir(subdf, atributos_restantes, profundidad + 1)
            nodo.hijos[valor] = hijo

        return nodo

    # ── Predicción ────────────────────────────────────────────────
    def predict(self, df):
        return df.apply(self._predecir_fila, axis=1).values

    def _predecir_fila(self, fila):
        nodo = self.raiz
        while not nodo.es_hoja:
            valor = fila.get(nodo.atributo)
            if valor not in nodo.hijos:
                break  # valor no visto → clase mayoritaria del nodo actual
            nodo = nodo.hijos[valor]
        return nodo.clase

    # ── Visualización ─────────────────────────────────────────────
    def export_text(self):
        """Imprime el árbol como reglas de decisión legibles."""
        if self.raiz is None:
            print("El árbol no fue entrenado todavía.")
            return
        lines = []
        self._texto_nodo(self.raiz, lines, prefijo="", rama="")
        print("\n".join(lines))

    def _texto_nodo(self, nodo, lines, prefijo, rama):
        encabezado = f"{prefijo}{rama}" if rama else prefijo
        if nodo.es_hoja:
            lines.append(f"{encabezado}→ {nodo.clase}  {nodo.conteos}")
            return
        lines.append(f"{encabezado}[{nodo.atributo}]")
        hijos = list(nodo.hijos.items())
        for i, (valor, hijo) in enumerate(hijos):
            es_ultimo = (i == len(hijos) - 1)
            conector   = "└── " if es_ultimo else "├── "
            continuador = "    " if es_ultimo else "│   "
            self._texto_nodo(hijo, lines,
                             prefijo=prefijo + continuador,
                             rama=f"{conector}{valor}: ")

# ── Smoke test con dataset toy (prueba que el árbol funciona) ─────
from sklearn.metrics import accuracy_score

_toy = pd.DataFrame({
    "clima":  ["sol","sol","nube","lluvia","lluvia","lluvia","nube","sol"],
    "viento": ["no", "si", "no",  "no",   "si",    "no",   "si",  "no"],
    "jugar":  ["si", "si", "si",  "si",   "no",    "no",   "no",  "no"],
})
_arbol_toy = ArbolC45(min_instancias=1)
_arbol_toy.fit(_toy, "jugar")
_pred_toy = _arbol_toy.predict(_toy)
print(f"✓ ArbolC45 smoke test OK")
print(f"  Predicciones : {list(_pred_toy)}")
print(f"  Accuracy toy : {accuracy_score(_toy['jugar'], _pred_toy):.2f}")

# ── Prueba con el dataset real si está disponible y tiene categorías ──
if "df" in dir() and "TARGET_COL" in dir():
    _cols_cat = atributos_categoricos(df, TARGET_COL)
    if _cols_cat:
        from sklearn.model_selection import train_test_split
        _df_cat = df[_cols_cat + [TARGET_COL]].dropna()
        X_train, X_test = train_test_split(_df_cat, test_size=0.2,
                                            random_state=SEED,
                                            stratify=_df_cat[TARGET_COL])
        arbol = ArbolC45(min_instancias=2)
        arbol.fit(X_train, TARGET_COL)
        acc = accuracy_score(X_test[TARGET_COL], arbol.predict(X_test))
        print(f"\n✓ ArbolC45 con dataset real")
        print(f"  Atributos : {_cols_cat}")
        print(f"  Accuracy  : {acc:.4f}")
    else:
        print("\nℹ️  Dataset real sin columnas categóricas (continuos → tarea 21)")


✓ ArbolC45 smoke test OK


  Predicciones : ['no', 'si', 'si', 'no', 'no', 'no', 'no', 'no']


  Accuracy toy : 0.75



✓ ArbolC45 con dataset real
  Atributos : ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
  Accuracy  : 0.7167


In [10]:
# El diálogo de Drive frena un Run all. El CSV se lee en la celda 1.2.
print("Sin montar Drive. Datos: data/dataset.csv o el respaldo Breast Cancer.")


Sin montar Drive. Datos: data/dataset.csv o el respaldo Breast Cancer.


---
# 3. Completar C4.5 (parte 2)

Acá el árbol deja de ser “demo de discretos” y se acerca a lo que pide un C4.5 usable de verdad.

### 3.1 Atributos continuos

**Slot: 3º** · Card 21

**Qué es:** un numérico no se parte "por cada valor", sino por un **umbral** t: a la izquierda `<= t`, a la derecha `> t`. Se prueban puntos medios entre valores ordenados y se elige el de mayor ganancia de información.

**Por qué:** casi todos los datasets reales tienen edades, dosis, medidas, etc. Sin esto el modelo queda a medias.

**Hacer:** `best_split_continuous` y registrarlo en el mismo dispatch de la §2.3. Documentar si evalúan *todos* los puntos medios o solo donde cambia la clase.

Esta celda implementa `evaluar_atributo_continuo()`, que permite al árbol C4.5 trabajar con atributos numéricos continuos. En lugar de abrir una rama por cada valor distinto, que con cientos de valores equivaldría a memorizar los datos, se busca un único punto de corte y se divide en dos ramas. El procedimiento ordena los valores del atributo, toma como candidatos los puntos medios entre valores consecutivos, calcula la ganancia de información de cada uno y se queda con el de mayor ganancia. La función devuelve el umbral, su ganancia y su gain ratio. `best_split_continuous(df, atributo, target)` la expone con la misma interfaz que `gain()` de la §2.2, y `atributos_continuos()` complementa a `atributos_categoricos()` de la §2.3.

**Decisión de diseño: solo puntos frontera.** Se evalúan solo los puntos medios donde cambia la clase, no todos. Fayyad e Irani (1992) demostraron que el umbral de máxima ganancia de información siempre se encuentra en uno de esos puntos, por lo que el resultado es el mismo con menos cálculo. En Breast Cancer, los 30 umbrales elegidos fueron idénticos en ambos modos, evaluando 3.527 candidatos en lugar de 11.002. El modo completo queda disponible con `solo_cambios_clase=False`.

**Resultado con el dataset de respaldo**

* El mejor corte es `worst perimeter <= 112.80`, con gain ratio 0,628.
* Ese solo corte clasifica correctamente el 89,5 % del conjunto de prueba (30 %, estratificado), por encima de la línea base de 62,7 %.

**Reglas del equipo**

* El árbol usa esta función para todo atributo numérico. Los categóricos siguen con su tratamiento de la §2.3.
* Rama izquierda = `<= umbral`, rama derecha = `> umbral` (convención acordada en la §2.1).
* Después de un corte el atributo sale de la lista, también si es continuo: en cada rama hay un solo umbral. El código hace `restantes = [a for a in atributos if a != atributo]`.
* El umbral se elige por ganancia y los atributos se comparan entre sí por gain ratio. No mezclar los criterios.
* Si la función devuelve `None`, el atributo no se puede usar en ese nodo.
* Los valores faltantes se ignoran en esta función. Su tratamiento corresponde a la §3.2.

In [11]:
# ═══ 3.1 Atributos continuos: búsqueda del umbral (Card 21) ═══
#
# DECISIÓN DEL EQUIPO: se evalúan SOLO los puntos medios donde cambia la clase
# ("puntos frontera"), no todos los puntos medios.
# Fundamento: Fayyad e Irani (1992) demostraron que el umbral de máxima ganancia de
# información siempre está en un punto frontera. Verificado en Breast Cancer: los 30
# umbrales elegidos son idénticos en ambos modos, con muchos menos candidatos.
# Un punto es frontera salvo que los dos grupos de valores vecinos sean puros y de la
# misma clase. El modo "todos" queda disponible con solo_cambios_clase=False.
#
# El umbral se elige por GANANCIA (como C4.5); el gain_ratio que se devuelve es el que
# usa el árbol para comparar este atributo contra los demás.


def _entropia_conteos(conteos):
    """Entropía (bits) a partir de conteos por clase. Acepta un vector o una matriz (una fila por caso)."""
    conteos = np.asarray(conteos, dtype=float)
    totales = conteos.sum(axis=-1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(totales > 0, conteos / totales, 0.0)
        terminos = np.where(p > 0, p * np.log2(p), 0.0)
    return -terminos.sum(axis=-1)


def evaluar_atributo_continuo(valores, y, solo_cambios_clase=True):
    """
    Busca el mejor umbral binario (valor <= umbral / valor > umbral) para un atributo continuo.

    Devuelve None si no hay corte posible (un solo valor distinto, ningún punto frontera
    o ganancia nula). Si no, un dict con:
        umbral      punto medio elegido
        ganancia    ganancia de información del corte (criterio para elegir el umbral)
        gain_ratio  ganancia / split info (criterio para comparar contra otros atributos)
        candidatos  cantidad de umbrales evaluados
    Los valores faltantes (NaN) se ignoran acá; su tratamiento es parte de la §3.2.
    """
    valores = np.asarray(valores, dtype=float)
    y = np.asarray(y)
    conocidos = ~np.isnan(valores)
    valores, y = valores[conocidos], y[conocidos]
    n = len(y)

    if n == 0 or np.unique(valores).size < 2:
        return None

    # Ordenar y acumular conteos por clase: acum[i] = clases vistas hasta la posición i
    orden = np.argsort(valores, kind="mergesort")
    v, ys = valores[orden], y[orden]
    clases, y_idx = np.unique(ys, return_inverse=True)
    acum = np.cumsum(np.eye(len(clases))[y_idx], axis=0)
    total = acum[-1]

    # Cortes posibles: después del último elemento de cada valor distinto
    fin = np.flatnonzero(v[1:] != v[:-1])
    izq = acum[fin]
    der = total - izq

    if solo_cambios_clase:
        finales = np.append(fin, n - 1)
        por_grupo = np.diff(np.vstack([np.zeros(len(clases)), acum[finales]]), axis=0)
        puro = np.where((por_grupo > 0).sum(axis=1) == 1, por_grupo.argmax(axis=1), -1)
        es_frontera = ~((puro[:-1] >= 0) & (puro[:-1] == puro[1:]))
        fin, izq, der = fin[es_frontera], izq[es_frontera], der[es_frontera]
        if fin.size == 0:
            return None

    # Ganancia de cada candidato, calculada para todos a la vez
    n_izq, n_der = izq.sum(axis=1), der.sum(axis=1)
    entropia_hijos = (n_izq * _entropia_conteos(izq) + n_der * _entropia_conteos(der)) / n
    ganancias = _entropia_conteos(total) - entropia_hijos

    mejor = int(np.argmax(ganancias))
    if ganancias[mejor] <= 0:
        return None

    umbral = (v[fin[mejor]] + v[fin[mejor] + 1]) / 2
    split_info = _entropia_conteos([n_izq[mejor], n_der[mejor]])

    return {
        "umbral": float(umbral),
        "ganancia": float(ganancias[mejor]),
        "gain_ratio": float(ganancias[mejor] / split_info),
        "candidatos": int(fin.size),
    }


def best_split_continuous(df, atributo, target, solo_cambios_clase=True):
    """Misma interfaz que gain(df, atributo, target) de la §2.2: es lo que llama el árbol."""
    return evaluar_atributo_continuo(df[atributo], df[target], solo_cambios_clase)


def atributos_continuos(df, target):
    """Columnas numéricas, excluyendo el target (complemento de atributos_categoricos de la §2.3)."""
    return [c for c in df.columns
            if c != target and pd.api.types.is_numeric_dtype(df[c])]


# ── Verificación rápida ───────────────────────────────────────────
assert evaluar_atributo_continuo([5, 5, 5], ["a", "b", "a"]) is None, "un solo valor → None"
_r = evaluar_atributo_continuo([1, 2, 3, 10, 11, 12], ["a", "a", "a", "b", "b", "b"])
assert _r["umbral"] == 6.5 and _r["ganancia"] == 1.0, "corte perfecto en 6.5"
print("✓ evaluar_atributo_continuo OK")

# ── Prueba con el dataset real ────────────────────────────────────
# El 0.85 era del corte único en Breast Cancer. Acá se informa el número.
_cols_cont = atributos_continuos(df, TARGET_COL)
if _cols_cont:
    _train, _test = train_test_split(df, test_size=0.3, random_state=SEED,
                                     stratify=df[TARGET_COL])

    _res = {c: best_split_continuous(_train, c, TARGET_COL) for c in _cols_cont}
    _res = {c: r for c, r in _res.items() if r is not None}
    _mejor = max(_res, key=lambda c: _res[c]["gain_ratio"])
    _u = _res[_mejor]["umbral"]

    # Un solo corte (stump): cada lado predice su clase mayoritaria
    _izq = _train.loc[_train[_mejor] <= _u, TARGET_COL].mode()[0]
    _der = _train.loc[_train[_mejor] > _u, TARGET_COL].mode()[0]
    _pred = np.where(_test[_mejor] <= _u, _izq, _der)
    _acc = (_pred == _test[TARGET_COL].values).mean()
    # No se exige 0.85: ese piso era de un atributo de Breast Cancer.

    # Comparación de los dos modos (dato para el informe)
    _todos = {c: best_split_continuous(_train, c, TARGET_COL, solo_cambios_clase=False)
              for c in _res}
    _iguales = all(abs(_res[c]["umbral"] - _todos[c]["umbral"]) < 1e-9 for c in _res)

    print(f"\n✓ Umbrales sobre el dataset real")
    print(f"  Mejor corte : {_mejor} <= {_u:.2f}  (gain ratio {_res[_mejor]['gain_ratio']:.3f})")
    print(f"  Accuracy    : {_acc:.4f}  (un solo corte, test 30 %)")
    print(f"  Candidatos  : {sum(r['candidatos'] for r in _res.values())} (solo cambios de clase) vs "
          f"{sum(r['candidatos'] for r in _todos.values())} (todos) · umbrales iguales: {_iguales}")
else:
    print("\nℹ️  Dataset real sin columnas numéricas")

✓ evaluar_atributo_continuo OK

✓ Umbrales sobre el dataset real
  Mejor corte : chol <= 42.50  (gain ratio 0.122)
  Accuracy    : 0.5978  (un solo corte, test 30 %)
  Candidatos  : 374 (solo cambios de clase) vs 454 (todos) · umbrales iguales: True


### 3.2 Parada, faltantes, poda y API sklearn

Esta sección presenta los criterios parametrizables, el tratamiento de valores faltantes, la estimación pesimista y la poda postorden, y su exposición mediante un estimador compatible con scikit-learn.

La notación distingue `CF`, el factor de confianza de la estimación pesimista, de `t`, la proporción de peso conocido para el atributo que divide cada nodo. Se almacenan por nodo y cumplen funciones diferentes.

---

### Auxiliar estadística para la estimación pesimista: `u(e, N, CF)`

La poda pesimista estima el error de generalización a partir de los errores observados. En un nodo, `N` es la masa total de casos y `e` la masa que no pertenece a la clase mayoritaria. Con valores faltantes, ambas cantidades pueden ser fraccionarias. La función `u(e, N, CF)` devuelve errores adicionales; el error total estimado es `e + u(e, N, CF)`, y la tasa correspondiente es `U_CF(e, N) = (e + u(e, N, CF)) / N`.

La rutina `AddErrs` del apéndice de Quinlan (1993, pp. 278-279) usa esta tabla de pares `CF` y `z`, con interpolación lineal:

| CF | z |
|---:|---:|
| 0.000 | 4.00 |
| 0.001 | 3.09 |
| 0.005 | 2.58 |
| 0.010 | 2.33 |
| 0.050 | 1.65 |
| 0.100 | 1.28 |
| 0.200 | 0.84 |
| 0.400 | 0.25 |
| 1.000 | 0.00 |

**Convención del listado.** La nota al pie 6 de la sección 4.2 (p. 41) presenta la cota con una probabilidad de cola `CF/2`. El listado de `AddErrs`, en cambio, interpola `z` directamente con `CF`. Para `CF=0.25`, esa tabla da `z=0.6925`; no es el cuantil `z≈1.15` que se obtiene al traducir `CF/2` a una cola normal. La implementación sigue la convención del listado y no mezcla ambas definiciones.

La rutina separa cuatro casos:

1. Si `e < 10^-6`, devuelve `N * (1 - CF^(1/N))`. Para `e=0`, `N=6` y `CF=0.25`, son aproximadamente `1.238` errores adicionales.
2. Si `e < 0.9999`, interpola linealmente entre el resultado para cero errores y el resultado para un error observado.
3. Si `e + 0.5 >= N`, aplica el extremo superior empírico `0.67 * (N - e)`.
4. En los demás casos, usa la cota normal con corrección de continuidad. Definiendo `q=(e+0.5)/N` y `z` según la tabla:

   `p_upper = (q + z^2/(2N) + z*sqrt(q/N - q^2/N + z^2/(4N^2))) / (1 + z^2/N)`

   `u(e,N,CF) = N*p_upper - e`.

**Referencia:** Quinlan, J. Ross. *C4.5: Programs for Machine Learning*. Morgan Kaufmann, 1993, sección 4.2, p. 41, nota 6, y listado de `AddErrs` en el apéndice, pp. 278-279.

La poda compara, en postorden, el error estimado de convertir un nodo en hoja con la suma de los errores estimados de sus hijos. La implementación de `u()` conserva la estructura y las constantes del listado citado; esto no implica que el árbol completo reproduzca todas las decisiones del programa original.

In [12]:
# Quinlan C4.5 (1993), rutina AddErrs del apendice, pp. 278-279.
# Devuelve errores adicionales estimados, no la tasa ni el error total.
# Esta tabla pertenece a AddErrs: z se interpola directamente sobre CF.
# No sustituirla por el z derivado de la nota al pie sobre CF/2 sin cambiar la convención.

# Tabla de coeficientes de AddErrs; cada z corresponde al CF de la misma posición.
# Para CF=0.25 se interpola entre 0.20 y 0.40: z=0.6925.
_C45_CF = np.array([0.0, 0.001, 0.005, 0.01, 0.05, 0.10, 0.20, 0.40, 1.0])
_C45_Z  = np.array([4.0, 3.09, 2.58, 2.33, 1.65, 1.28, 0.84, 0.25, 0.0])


def u(e, N, CF=0.25):
    """Errores adicionales estimados por la rutina de poda de C4.5.

    e: errores observados en el nodo (puede ser fraccional por pesos).
    N: instancias/peso total del nodo.
    CF: factor de confianza, por defecto 0.25.
    Devuelve los errores adicionales; el total estimado es e + u(...).
    """
    e = float(e)
    N = float(N)
    CF = float(CF)

    if not np.isfinite(e) or not np.isfinite(N) or not np.isfinite(CF):
        raise ValueError("e, N y CF deben ser finitos.")
    if N <= 0:
        raise ValueError("N debe ser mayor que cero.")
    if e < 0 or e > N:
        raise ValueError("e debe estar entre 0 y N.")
    if not 0 < CF <= 1:
        raise ValueError("CF debe estar en el intervalo (0, 1].")

    # C4.5 interpola z respecto de CF y usa su cuadrado como coeficiente.
    z = float(np.interp(CF, _C45_CF, _C45_Z))
    coef = z * z

    # Caso base: cero errores observados; el resultado sigue siendo cantidad de errores.
    base = N * (1.0 - CF ** (1.0 / N))
    if e < 1e-6:
        return base

    # C4.5 interpola linealmente para errores entre cero y uno.
    if e < 0.9999:
        add_at_one = u(1.0, N, CF)
        return base + e * (add_at_one - base)

    # Tratamiento del extremo superior con correccion de continuidad.
    if e + 0.5 >= N:
        return 0.67 * (N - e)

    f = (e + 0.5) / N
    p_upper = (
        f
        + coef / (2.0 * N)
        + z * np.sqrt(
            f / N - (f * f) / N + coef / (4.0 * N * N)
        )
    ) / (1.0 + coef / N)

    return N * p_upper - e


def U_cf(e, N, CF=0.25):
    """Tasa pesimista, distinta de u(), que devuelve errores adicionales."""
    return (e + u(e, N, CF)) / N


# Para CF=0.25, la tabla da z ~= 0.6925 (interpolacion entre 0.20 y 0.40).

### Criterios de parada parametrizables

La clase auxiliar `_ArbolC45Parametrizable` aplica estos parámetros durante la construcción. `DecisionTreeC45` los expone mediante la interfaz de scikit-learn. El árbol integra los cortes numéricos de la sección 3.1 y evalúa los candidatos con masas ponderadas para conservar el manejo fraccional de faltantes.

| Parámetro | Efecto |
|---|---|
| `max_depth` | Profundidad máxima; la raíz tiene profundidad 0. `max_depth=1` permite una decisión en la raíz; `max_depth=3`, hasta tres niveles de decisión. |
| `min_samples_split` | Masa mínima de ejemplos que debe alcanzar un nodo para intentar dividirlo. |
| `min_samples_leaf` | Masa mínima esperada de cada rama después del reparto fraccional; si alguna no alcanza el mínimo, se descarta ese atributo. |
| `min_gain_ratio` | Rechaza divisiones cuya ganancia relativa no alcanza el umbral. |
| `random_state` | Hace reproducible la elección entre atributos empatados. |
| `pruning`, `pruning_confidence` | Si `pruning=True`, aplica PEP después del crecimiento; `pruning_confidence` se usa como `CF`. |

Los chequeos se aplican en este orden: clase → sin atributos utilizables → tamaño → profundidad → ganancia. Esta implementación usa atributos categóricos y continuos; aplica los parámetros sobre masas ponderadas cuando hay valores faltantes.

In [13]:
class _ArbolC45Parametrizable(ArbolC45):
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1,
                 min_gain_ratio=0.0, pruning=False, pruning_confidence=0.25,
                 random_state=None):
        if max_depth is not None and max_depth < 0:
            raise ValueError("max_depth debe ser >= 0 o None.")
        if min_samples_split < 2:
            raise ValueError("min_samples_split debe ser >= 2.")
        if min_samples_leaf < 1:
            raise ValueError("min_samples_leaf debe ser >= 1.")
        if min_gain_ratio < 0:
            raise ValueError("min_gain_ratio debe ser >= 0.")
        if not 0 < pruning_confidence <= 1:
            raise ValueError("pruning_confidence debe estar en (0, 1].")
        super().__init__(min_instancias=min_samples_split, profundidad_max=max_depth)
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_gain_ratio = min_gain_ratio
        self.pruning = pruning
        self.pruning_confidence = pruning_confidence
        self.random_state = random_state
        self._rng = None

    @staticmethod
    def _masa(pesos):
        return float(np.asarray(pesos, dtype=float).sum())

    def fit(self, df, target):
        if not isinstance(df, pd.DataFrame):
            raise TypeError("df debe ser un DataFrame.")
        if len(df) == 0:
            raise ValueError("No se puede entrenar con un DataFrame vacío.")
        if target not in df.columns:
            raise ValueError(f"No existe la columna target {target!r}.")
        if df[target].isna().any():
            raise ValueError("La columna target no puede contener valores faltantes.")
        self.target = target
        # Integra atributos categóricos y numéricos; estos últimos usan cortes de la sección 3.1.
        self.atributos_ = [
            c for c in df.columns if c != target and (
                (pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c]))
                or pd.api.types.is_bool_dtype(df[c])
                or df[c].dtype in ("object", "category")
                or _es_categorica(df[c])
            )
        ]
        if not self.atributos_:
            raise ValueError("Se requiere al menos un atributo categórico o numérico.")
        datos = df.reset_index(drop=True).copy()
        self.clases_ = list(np.unique(datos[target].to_numpy()))
        self._rng = np.random.default_rng(self.random_state)
        self.raiz = self._construir(datos, np.ones(len(datos)), self.atributos_, 0)
        self.error_estimado_ = None
        if self.pruning:
            self.error_estimado_ = podar_pesimista(self.raiz, self.pruning_confidence)
        return self

    def _candidatos(self, df, pesos, atributos):
        """Evalúa cortes categóricos y continuos con masas ponderadas.

        Para continuos aplica la decisión de 3.1: puntos medios entre grupos
        vecinos, salvo si ambos son puros y de la misma clase.
        """
        y = df[self.target].to_numpy()
        masa_total = self._masa(pesos)
        candidatos = []
        for atributo in atributos:
            serie = df[atributo]
            conocidos = serie.notna().to_numpy()
            masa_conocida = self._masa(pesos[conocidos])
            if masa_conocida <= 0:
                continue
            masa_faltante = masa_total - masa_conocida
            continuo = (pd.api.types.is_numeric_dtype(serie)
                        and not pd.api.types.is_bool_dtype(serie))
            if continuo:
                valores = serie.to_numpy(dtype=float, na_value=np.nan)
                posiciones = np.flatnonzero(conocidos)
                orden = posiciones[np.argsort(valores[posiciones], kind="mergesort")]
                v = valores[orden]
                cortes = np.flatnonzero(v[1:] != v[:-1])
                if cortes.size == 0:
                    continue
                inicios = np.r_[0, cortes + 1]
                finales = np.r_[cortes, len(orden) - 1]
                grupos = np.zeros((len(inicios), len(self.clases_)), dtype=float)
                for gi, (ini, fin) in enumerate(zip(inicios, finales)):
                    idx = orden[ini:fin + 1]
                    for ci, clase in enumerate(self.clases_):
                        grupos[gi, ci] = self._masa(pesos[idx][y[idx] == clase])
                presentes = grupos > 0
                puros = np.where(presentes.sum(axis=1) == 1, presentes.argmax(axis=1), -1)
                fronteras = ~((puros[:-1] >= 0) & (puros[:-1] == puros[1:]))
                for gi in np.flatnonzero(fronteras):
                    umbral = float((v[cortes[gi]] + v[cortes[gi] + 1]) / 2)
                    masks = [conocidos & (valores <= umbral), conocidos & (valores > umbral)]
                    masas = np.asarray([self._masa(pesos[m]) for m in masks])
                    if np.any(masas <= 0):
                        continue
                    hp = _entropia_ponderada(y[conocidos], pesos[conocidos])
                    hh = sum(masas[j] / masa_conocida * _entropia_ponderada(y[masks[j]], pesos[masks[j]]) for j in range(2))
                    ganancia = masa_conocida / masa_total * (hp - hh)
                    split = np.append(masas, masa_faltante) if masa_faltante > 0 else masas
                    ps = split / split.sum()
                    si = float(-np.sum(ps[ps > 0] * np.log2(ps[ps > 0])))
                    ratio = ganancia / si if si > 0 else 0.0
                    esperadas = masas + masa_faltante * masas / masa_conocida
                    if np.any(esperadas < self.min_samples_leaf):
                        continue
                    if ratio >= self.min_gain_ratio:
                        candidatos.append((atributo, ratio, ["<=", ">"], masks, masas,
                                           masa_conocida, masa_total, True, umbral))
            else:
                ramas = list(pd.unique(serie[conocidos]))
                masks = [conocidos & serie.eq(v).fillna(False).to_numpy() for v in ramas]
                masas = np.asarray([self._masa(pesos[m]) for m in masks])
                if len(masas) < 2 or np.any(masas <= 0):
                    continue
                hp = _entropia_ponderada(y[conocidos], pesos[conocidos])
                hh = sum(masas[j] / masa_conocida * _entropia_ponderada(y[masks[j]], pesos[masks[j]]) for j in range(len(masas)))
                ganancia = masa_conocida / masa_total * (hp - hh)
                split = np.append(masas, masa_faltante) if masa_faltante > 0 else masas
                ps = split / split.sum()
                si = float(-np.sum(ps[ps > 0] * np.log2(ps[ps > 0])))
                ratio = ganancia / si if si > 0 else 0.0
                esperadas = masas + masa_faltante * masas / masa_conocida
                if np.any(esperadas < self.min_samples_leaf):
                    continue
                if ratio >= self.min_gain_ratio:
                    candidatos.append((atributo, ratio, ramas, masks, masas,
                                       masa_conocida, masa_total, False, None))
        return candidatos

    def _construir(self, df, pesos, atributos, profundidad):
        nodo = Nodo()
        nodo.conteos = _conteos_ponderados(df[self.target].to_numpy(), pesos)
        nodo.peso_total = self._masa(pesos)
        nodo.n_total = nodo.peso_total
        nodo.clase = max(nodo.conteos, key=nodo.conteos.get)
        nodo.es_hoja = True
        nodo.t = 1.0
        nodo.n_conocidos = nodo.n_total
        nodo.ramas_pesos = {}
        if len(nodo.conteos) <= 1 or not atributos:
            return nodo
        if nodo.n_total < self.min_samples_split:
            return nodo
        if self.max_depth is not None and profundidad >= self.max_depth:
            return nodo

        candidatos = self._candidatos(df, pesos, atributos)
        if not candidatos:
            return nodo
        max_ratio = max(item[1] for item in candidatos)
        empatados = [item for item in candidatos
                     if np.isclose(item[1], max_ratio, rtol=1e-12, atol=1e-15)]
        elegido = empatados[int(self._rng.integers(len(empatados)))] if len(empatados) > 1 else empatados[0]
        (atributo, _, ramas, mascaras, masas, masa_conocida, _,
         es_continuo, umbral) = elegido
        conocidos = df[atributo].notna().to_numpy()
        faltantes = ~conocidos
        nodo.es_hoja = False
        nodo.atributo = atributo
        nodo.es_continuo = es_continuo
        nodo.umbral = umbral
        nodo.n_conocidos = masa_conocida
        nodo.t = masa_conocida / nodo.n_total
        restantes = [a for a in atributos if a != atributo]
        for rama, mascara, masa_rama in zip(ramas, mascaras, masas):
            prior = float(masa_rama / masa_conocida)
            posiciones = np.flatnonzero(mascara)
            df_hijo = df.iloc[posiciones].copy()
            pesos_hijo = pesos[posiciones].copy()
            if faltantes.any():
                df_hijo = pd.concat([df_hijo, df.loc[faltantes]], ignore_index=True)
                pesos_hijo = np.concatenate([pesos_hijo, pesos[faltantes] * prior])
            else:
                df_hijo = df_hijo.reset_index(drop=True)
            nodo.hijos[rama] = self._construir(
                df_hijo.reset_index(drop=True), pesos_hijo, restantes, profundidad + 1
            )
            nodo.ramas_pesos[rama] = prior
        return nodo

    def predict_proba(self, X):
        if self.raiz is None:
            raise RuntimeError("El árbol debe entrenarse antes de predecir.")
        clases = self.clases_
        def distribuir(nodo, fila):
            if nodo.es_hoja:
                return _normalizar_conteos(nodo.conteos, clases)
            valor = fila.get(nodo.atributo, np.nan)
            if pd.notna(valor):
                if getattr(nodo, "es_continuo", False):
                    rama = "<=" if float(valor) <= nodo.umbral else ">"
                    return distribuir(nodo.hijos[rama], fila)
                if valor in nodo.hijos:
                    return distribuir(nodo.hijos[valor], fila)
            salida = {clase: 0.0 for clase in clases}
            for rama, hijo in nodo.hijos.items():
                proba = distribuir(hijo, fila)
                for clase in clases:
                    salida[clase] += nodo.ramas_pesos[rama] * proba[clase]
            return salida
        filas = [distribuir(self.raiz, fila) for _, fila in X.iterrows()]
        return np.asarray([[fila[c] for c in clases] for fila in filas], dtype=float)

    def predict(self, X):
        proba = self.predict_proba(X)
        pred = np.asarray(self.clases_)[np.argmax(proba, axis=1)]
        # sklearn trata un array object de números como target desconocido.
        if pred.dtype == object:
            pred = np.asarray(pred.tolist())
        return pred


### Manejo fraccional de valores faltantes

Cuando falta el valor del atributo elegido para dividir, el caso se distribuye entre las ramas conocidas. Para cada rama, su peso se multiplica por `masa_conocida_rama / masa_conocida_total`. Las proporciones se calculan con los pesos que llegan al nodo y se conservan sin redondeo.

El entrenamiento usa los pesos para evaluar las divisiones, calcular entropías y conteos, y construir recursivamente los hijos. En predicción, un valor faltante o no observado combina las distribuciones de las ramas según sus proporciones conocidas.

In [14]:
def _conteos_ponderados(y, pesos):
    y = np.asarray(y)
    pesos = np.asarray(pesos, dtype=float)
    if len(y) != len(pesos):
        raise ValueError("y y pesos deben tener la misma longitud.")
    if np.any(~np.isfinite(pesos)) or np.any(pesos < 0):
        raise ValueError("Los pesos deben ser finitos y no negativos.")
    tabla = pd.DataFrame({"clase": y, "peso": pesos})
    tabla = tabla[tabla["clase"].notna() & (tabla["peso"] > 0)]
    return tabla.groupby("clase", sort=False)["peso"].sum().to_dict()


def _entropia_ponderada(y, pesos):
    conteos = _conteos_ponderados(y, pesos)
    total = float(sum(conteos.values()))
    if total == 0:
        return 0.0
    p = np.asarray(list(conteos.values()), dtype=float) / total
    return float(-np.sum(p * np.log2(p)))


def _normalizar_conteos(conteos, clases):
    total = float(sum(conteos.values()))
    return {c: (float(conteos.get(c, 0.0)) / total if total else 0.0) for c in clases}


def _recorrer_nodos_card24(nodo):
    yield nodo
    for hijo in nodo.hijos.values():
        yield from _recorrer_nodos_card24(hijo)


# El caso faltante se reparte entre ramas con masas conocidas 2/3 y 1/3.
_prueba_reparto = pd.DataFrame({"a": ["x", "x", "y", np.nan],
                                "clase": ["A", "A", "B", "B"]})
_arbol_fraccional = _ArbolC45Parametrizable(min_samples_split=2).fit(
    _prueba_reparto, "clase"
)
_raiz_fraccional = _arbol_fraccional.raiz
assert _raiz_fraccional.atributo == "a"
assert np.isclose(_raiz_fraccional.hijos["x"].conteos["B"], 2/3)
assert np.isclose(_raiz_fraccional.hijos["y"].conteos["B"], 4/3)
assert np.isclose(sum(h.peso_total for h in _raiz_fraccional.hijos.values()), 4.0)
assert np.isclose(_entropia_ponderada(["A", "B"], [3, 1]),
                   -(3/4)*np.log2(3/4) -(1/4)*np.log2(1/4))

_arbol_sin_faltante = _ArbolC45Parametrizable(min_samples_split=2).fit(
    _prueba_reparto.dropna(subset=["a"]).reset_index(drop=True), "clase"
)
_consulta_faltante = pd.DataFrame({"a": [np.nan]})
_proba_con_faltante = _arbol_fraccional.predict_proba(_consulta_faltante)[0]
_proba_fila_eliminada = _arbol_sin_faltante.predict_proba(_consulta_faltante)[0]
assert not np.allclose(_proba_con_faltante, _proba_fila_eliminada)
assert np.isclose(_proba_con_faltante.sum(), 1.0)
print("Reparto fraccional, entropía ponderada y comparación con fila eliminada verificados.")

Reparto fraccional, entropía ponderada y comparación con fila eliminada verificados.


### Factor de proporción conocida por nodo

Cada nodo de decisión almacena `n_total`, `n_conocidos` para el atributo elegido y `t = n_conocidos / n_total`. En las hojas, que no evalúan atributos, `t=1`. `t` mide la proporción de peso conocido; `CF` es el factor de confianza de la estimación pesimista y se configura por separado con `pruning_confidence`.

Durante PEP, `t` ajusta el tamaño efectivo usado en la cota. Para conservar la proporción observada de errores, se usan `N_efectivo = max(1, t * n_total)` y `e_efectivo = (e / n_total) * N_efectivo`; la tasa pesimista resultante se proyecta sobre `n_total`. El mínimo de una unidad evita aplicar la interpolación de `AddErrs` fuera del dominio de su rama para errores fraccionales. Con datos completos, `t=1` y la corrección no altera el tamaño ni la tasa observada. `t` y `CF` son factores distintos.

In [15]:
# Humo 1: con atributos completos, cada nodo conserva t=1.0.
_arbol_completo_t = _ArbolC45Parametrizable(min_samples_split=2).fit(
    pd.DataFrame({"color": ["rojo", "rojo", "azul", "azul"],
                  "clase": ["si", "si", "no", "no"]}), "clase"
)
_nodos_completos_t = list(_recorrer_nodos_card24(_arbol_completo_t.raiz))
assert all(np.isclose(n.t, 1.0) for n in _nodos_completos_t)

# Humo 2: la raíz guarda la fracción conocida de su atributo de división.
_arbol_faltante_t = _ArbolC45Parametrizable(min_samples_split=2).fit(
    pd.DataFrame({"color": ["rojo", "rojo", "azul", np.nan],
                  "clase": ["si", "si", "no", "no"]}), "clase"
)
_raiz_t = _arbol_faltante_t.raiz
assert _raiz_t.atributo == "color"
assert np.isclose(_raiz_t.n_total, 4.0)
assert np.isclose(_raiz_t.n_conocidos, 3.0)
assert np.isclose(_raiz_t.t, 0.75)
print("t=1.0 con datos completos y t=0.75 con un faltante en la raíz.")

t=1.0 con datos completos y t=0.75 con un faltante en la raíz.


### Poda pesimista (PEP)

PEP visita el árbol en postorden: primero procesa los hijos y después compara cada nodo. Para cada nodo calcula `e`, la masa que no pertenece a su clase mayoritaria, y `n`, la masa total. En los nodos de decisión aplica `t` al tamaño efectivo y conserva la tasa observada al ajustar `e`. La tasa pesimista se proyecta sobre la masa del nodo.

`error_hoja` es el error estimado al reemplazar el subárbol por una hoja; `error_subárbol` es la suma de los errores estimados de sus hijos. Si `error_hoja <= error_subárbol`, el nodo se convierte en hoja. Como `u()` devuelve errores adicionales, cada error total se calcula como `e + u(e, N, CF)`. La comparación de tamaño y métricas informa resultados de una partición concreta, no garantiza una mejora universal de accuracy.

In [16]:
def _error_pesimista_nodo(nodo, CF):
    n = float(sum(nodo.conteos.values()))
    if n <= 0:
        return 0.0
    e = float(n - max(nodo.conteos.values()))
    t = float(getattr(nodo, "t", 1.0))
    n_efectivo = max(1.0, t * n)
    e_efectivo = (e / n) * n_efectivo
    tasa = (e_efectivo + u(e_efectivo, n_efectivo, CF)) / n_efectivo
    return n * tasa


def podar_pesimista(nodo, CF=0.25):
    """Poda en postorden y devuelve el error estimado total del nodo."""
    if nodo.es_hoja:
        return _error_pesimista_nodo(nodo, CF)
    error_subarbol = sum(podar_pesimista(hijo, CF) for hijo in nodo.hijos.values())
    error_hoja = _error_pesimista_nodo(nodo, CF)
    if error_hoja <= error_subarbol:
        nodo.es_hoja = True
        nodo.clase = max(nodo.conteos, key=nodo.conteos.get)
        nodo.atributo = None
        nodo.es_continuo = False
        nodo.umbral = None
        nodo.hijos.clear()
        nodo.ramas_pesos = {}
        return error_hoja
    return error_subarbol


class ArbolC45PEP(_ArbolC45Parametrizable):
    """árbol parametrizable que aplica PEP después del crecimiento."""
    def __init__(self, **kwargs):
        kwargs["pruning"] = True
        super().__init__(**kwargs)


def contar_nodos_arbol(nodo):
    """Cuenta los nodos para comparar el tamaño del árbol."""
    return 1 + sum(contar_nodos_arbol(hijo) for hijo in nodo.hijos.values())


def recorrer_nodos_arbol(nodo):
    yield nodo
    for hijo in nodo.hijos.values():
        yield from recorrer_nodos_arbol(hijo)

### Interoperabilidad con scikit-learn

`DecisionTreeC45` adapta el árbol parametrizable a la interfaz de scikit-learn. `BaseEstimator` proporciona `get_params`, `set_params` y la representación de parámetros; `ClassifierMixin` aporta `score`. El wrapper recibe un DataFrame `X` y el vector `y` por separado. El estimador admite atributos categóricos y numéricos; los numéricos se dividen por el umbral guardado en cada nodo

`predict_proba` devuelve las distribuciones de clase en el orden de `classes_`. Ante un valor faltante o no observado, combina las probabilidades de las ramas según las proporciones conocidas del nodo. `predict` elige la clase de mayor probabilidad, por lo que ambas interfaces resuelven esos casos de la misma manera.

El ejemplo comprueba `cross_val_score`, `GridSearchCV`, `predict_proba` y la suma de probabilidades por fila. La comparación de PEP informa tamaño, accuracy de entrenamiento y prueba, y brecha para una misma partición; esos resultados describen ese ejemplo y no garantizan una mejora universal.

In [17]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV


class DecisionTreeC45(ClassifierMixin, BaseEstimator):
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1,
                 min_gain_ratio=0.0, pruning=False, pruning_confidence=0.25,
                 random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_gain_ratio = min_gain_ratio
        self.pruning = pruning
        self.pruning_confidence = pruning_confidence
        self.random_state = random_state

    def fit(self, X, y):
        if globals().get("_AVISAR_FOLD"):
            _n = globals().get("_N_FOLD", 0) + 1
            globals()["_N_FOLD"] = _n
            print(f"    ajuste {_n}/25", flush=True)
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X debe ser un DataFrame.")
        y = np.asarray(y)
        if len(X) != len(y):
            raise ValueError("X e y deben tener la misma cantidad de filas.")
        self._target_col_ = "__clase_objetivo__"
        while self._target_col_ in X.columns:
            self._target_col_ += "_"
        datos = X.reset_index(drop=True).copy()
        datos[self._target_col_] = y
        self.arbol_ = _ArbolC45Parametrizable(
            max_depth=self.max_depth, min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf, min_gain_ratio=self.min_gain_ratio,
            pruning=self.pruning, pruning_confidence=self.pruning_confidence,
            random_state=self.random_state,
        ).fit(datos, self._target_col_)
        self.classes_ = np.asarray(self.arbol_.clases_)
        return self

    def predict_proba(self, X):
        if not hasattr(self, "arbol_"):
            raise RuntimeError("El estimador debe entrenarse antes de predecir.")
        return self.arbol_.predict_proba(X)

    def predict(self, X):
        proba = self.predict_proba(X)
        pred = self.classes_[np.argmax(proba, axis=1)]
        # Mismo criterio que arriba: números en dtype object rompen accuracy_score.
        if getattr(pred, "dtype", None) == object:
            pred = np.asarray(pred.tolist())
        return pred


_X_sklearn_demo = pd.DataFrame({
    "clima": ["sol", "sol", "nube", "lluvia", "lluvia", "lluvia", "nube", "sol"],
    "viento": ["no", "si", "no", "no", "si", "no", "si", "no"],
})
_y_sklearn_demo = np.array(["si", "si", "si", "si", "no", "no", "no", "no"])
_cv_sklearn_demo = StratifiedKFold(n_splits=2, shuffle=True, random_state=SEED)
_model_sklearn_demo = DecisionTreeC45(min_samples_split=2, max_depth=3)
assert isinstance(_model_sklearn_demo, BaseEstimator)
assert "max_depth=3" in repr(_model_sklearn_demo)
_scores_sklearn_demo = cross_val_score(
    DecisionTreeC45(min_samples_split=2), _X_sklearn_demo,
    _y_sklearn_demo, cv=_cv_sklearn_demo,
)
_grid_sklearn_demo = GridSearchCV(
    DecisionTreeC45(), {"min_samples_split": [2, 3], "max_depth": [None, 2]},
    cv=_cv_sklearn_demo,
).fit(_X_sklearn_demo, _y_sklearn_demo)
_model_elegido = _grid_sklearn_demo.best_estimator_
_proba_demo = _model_elegido.predict_proba(_X_sklearn_demo)
assert _proba_demo.shape == (len(_X_sklearn_demo), len(_model_elegido.classes_))
assert np.allclose(_proba_demo.sum(axis=1), 1.0)
_X_con_nan_demo = _X_sklearn_demo.iloc[[0]].copy()
_X_con_nan_demo.iloc[0, 0] = np.nan
_proba_nan_demo = _model_elegido.predict_proba(_X_con_nan_demo)
_pred_nan_demo = _model_elegido.predict(_X_con_nan_demo)
assert np.allclose(_proba_nan_demo.sum(axis=1), 1.0)
assert _pred_nan_demo[0] in _model_elegido.classes_
print("API sklearn y predict_proba configuradas", _scores_sklearn_demo)

_pep_X_train, _pep_X_test, _pep_y_train, _pep_y_test = train_test_split(
    _X_sklearn_demo, _y_sklearn_demo, test_size=0.25,
    random_state=SEED, stratify=_y_sklearn_demo,
)
for _pep_activa in (False, True):
    _pep_modelo = DecisionTreeC45(
        min_samples_split=2, pruning=_pep_activa,
        pruning_confidence=0.25, random_state=SEED,
    ).fit(_pep_X_train, _pep_y_train)
    _pep_acc_train = accuracy_score(_pep_y_train, _pep_modelo.predict(_pep_X_train))
    _pep_acc_test = accuracy_score(_pep_y_test, _pep_modelo.predict(_pep_X_test))
    print(
        f"PEP={_pep_activa}: nodos={contar_nodos_arbol(_pep_modelo.arbol_.raiz)}, "
        f"accuracy train={_pep_acc_train:.3f}, test={_pep_acc_test:.3f}, "
        f"brecha={_pep_acc_train - _pep_acc_test:.3f}"
    )


API sklearn y predict_proba configuradas [0.   0.25]
PEP=False: nodos=7, accuracy train=0.667, test=0.500, brecha=0.167
PEP=True: nodos=3, accuracy train=0.667, test=0.500, brecha=0.167


### Verificar que el núcleo no miente

Esta celda contiene seis pruebas pequeñas para la implementación final `DecisionTreeC45`. No usan ni reemplazan el dataset oficial: comprueban casos matemáticos y problemas de referencia.

Las tres primeras verifican la entropía calculada a mano. `entropy` recibe etiquetas, no un vector de conteos: dos clases distintas dan 1 bit, y una clase sola con otra categoría en cero da 0 sin `NaN`. Iris comprueba un caso multiclase conocido. XOR comprueba que el árbol pueda resolver una frontera que requiere al menos dos niveles. Por último, `sklearn.tree.DecisionTreeClassifier` sirve como referencia de cordura: se comparan resultados generales, sin exigir igualdad de árboles o predicciones, porque sklearn usa Gini y este modelo usa Gain Ratio.

Se usa `DecisionTreeC45`, que recibe `X` como DataFrame y `y` como vector. Esta celda queda después de la que define la clase, así un Run all la encuentra.


In [18]:
# Tests del núcleo de DecisionTreeC45 (sección 3.2).
# Cada bloque tiene un propósito distinto; un assert fallido indica qué propiedad revisar.

# 1-3. entropy() lee etiquetas, no conteos. [1, 1] serían dos filas de la misma clase.
# El caso puro usa una categoría con conteo 0 para comprobar que no aparece NaN.
assert np.isclose(entropy([0, 1]), 1.0), "Dos clases equiprobables deben tener entropía 1 bit."
assert np.isclose(entropy([0, 1, 2, 3]), 2.0), "Cuatro clases equiprobables deben tener entropía 2 bits."
_puro = pd.Series(pd.Categorical(["a"] * 10, categories=["a", "b"]))
assert np.isclose(entropy(_puro), 0.0), "Una distribución pura debe tener entropía 0, no NaN."
assert not np.isnan(entropy(_puro))

# La clase vive en la celda de arriba. Si el kernel se reinició, corré esa primero.
if "DecisionTreeC45" not in globals():
    raise RuntimeError("Falta DecisionTreeC45. Corré la celda que define la clase y después esta.")

# 4. Iris: problema multiclase conocido. Se ajusta y evalúa sobre el conjunto completo;
# esto es una prueba funcional, no una estimación de generalización.
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
_iris = load_iris(as_frame=True)
_X_iris = _iris.data
_y_iris = _iris.target
_dt_iris = DecisionTreeC45(random_state=SEED).fit(_X_iris, _y_iris)
_acc_iris = accuracy_score(_y_iris, _dt_iris.predict(_X_iris))
assert _acc_iris > 0.90, f"DecisionTreeC45 debe superar 0.90 de accuracy en Iris (obtuvo {_acc_iris:.3f})."

# 5. XOR: ninguna feature por si sola separa las clases. Se requieren dos niveles.
_X_xor = pd.DataFrame({"x1": [0, 0, 1, 1], "x2": [0, 1, 0, 1]})
_y_xor = np.array([0, 1, 1, 0])
_dt_xor = DecisionTreeC45(random_state=SEED).fit(_X_xor, _y_xor)
_acc_xor = accuracy_score(_y_xor, _dt_xor.predict(_X_xor))
def _profundidad_decision(nodo):
    """Cuenta niveles de decisión: hoja=0; una división en la raíz=1."""
    if nodo.es_hoja:
        return 0
    return 1 + max(_profundidad_decision(hijo) for hijo in nodo.hijos.values())

_prof_xor = _profundidad_decision(_dt_xor.arbol_.raiz)
assert _acc_xor == 1.0, f"El árbol debe clasificar los cuatro ejemplos XOR (accuracy {_acc_xor:.3f})."
assert _prof_xor >= 2, f"XOR requiere profundidad de decisión >= 2 (obtuvo {_prof_xor})."

# 6. Referencia de cordura con sklearn sobre Iris. No se comparan árboles ni
# se espera igualdad: sklearn usa Gini por defecto y DecisionTreeC45 usa Gain Ratio.
_dt_sklearn = DecisionTreeClassifier(max_depth=None, random_state=SEED).fit(_X_iris, _y_iris)
_acc_sklearn_iris = accuracy_score(_y_iris, _dt_sklearn.predict(_X_iris))
assert _acc_sklearn_iris > 0.90, f"La referencia sklearn debería superar 0.90 en Iris (obtuvo {_acc_sklearn_iris:.3f})."

print("✓ 6 tests del núcleo pasaron")
print(f"  Entropía: 1, 2 y 0 bits | Iris: C4.5={_acc_iris:.3f}, sklearn={_acc_sklearn_iris:.3f} (referencia, no equivalencia)")
print(f"  XOR: accuracy={_acc_xor:.1f}, profundidad={_prof_xor}")


✓ 6 tests del núcleo pasaron
  Entropía: 1, 2 y 0 bits | Iris: C4.5=0.973, sklearn=1.000 (referencia, no equivalencia)
  XOR: accuracy=1.0, profundidad=2


### Prueba de tamaño y efecto de la poda

Esta prueba compara tres configuraciones de `DecisionTreeC45` sobre la misma partición estratificada de entrenamiento y prueba: un árbol sin poda, uno con poda pesimista (`pruning=True`) y uno con `min_samples_leaf` grande. Así, las diferencias observadas se deben a la configuración del árbol y no a que cada variante haya recibido ejemplos distintos.

**Qué mide:**

- **Nodos, hojas y profundidad máxima:** describen el tamaño y la complejidad estructural del árbol. Los nodos incluyen decisiones y hojas; las hojas son los resultados finales; la profundidad es la cantidad de decisiones del camino más largo desde la raíz.
- **Accuracy de entrenamiento y de prueba:** proporción de predicciones correctas en los ejemplos usados para ajustar el árbol y en los reservados para evaluarlo, respectivamente.
- **Brecha train/test:** accuracy de entrenamiento menos accuracy de prueba. Una brecha menor indica que el rendimiento en entrenamiento y en datos no vistos está más cerca.
- **Coincidencia de predicciones:** proporción de ejemplos de prueba en que las variantes predicen la misma clase. La poda puede cambiar algunos resultados; la prueba comprueba que la mayoría se conserva.

La celda calcula las medidas recorriendo la raíz del modelo (`arbol_.raiz`), porque `DecisionTreeC45` no expone métodos públicos llamados `n_nodes()`, `n_leaves()` o `max_depth_()`. Las aserciones comprueban que la poda reduzca estrictamente el número de nodos. En este único corte, la accuracy de prueba y la brecha pueden moverse hasta 3 puntos: la poda decide con el entrenamiento, no con el holdout. Para `min_samples_leaf`, se exige un árbol menor y una accuracy de prueba dentro de tres puntos porcentuales de la referencia sin poda o superior. Ese margen operacionaliza “similar o mejor” para este test.

**Cómo leer un fallo:** si una aserción falla, la configuración o el conjunto de datos no cumple esa propiedad en esta partición. Conviene revisar la implementación y el resultado antes de concluir que la poda siempre se comporta así: esta prueba usa un único corte estratificado fijo para ser reproducible; la Tabla 5 de la sección 4 aporta la comparación agregada en validación cruzada.


In [19]:
# Usa una única partición estratificada y reproducible para las tres configuraciones.


def _medir_estructura_arbol(nodo, profundidad=0):
    """Devuelve (nodos, hojas, profundidad máxima) desde la raíz del árbol."""
    hijos = getattr(nodo, "hijos", {})
    if getattr(nodo, "es_hoja", False) or not hijos:
        return 1, 1, profundidad
    medidas = [
        _medir_estructura_arbol(hijo, profundidad + 1)
        for hijo in hijos.values()
    ]
    return (
        1 + sum(m[0] for m in medidas),
        sum(m[1] for m in medidas),
        max(m[2] for m in medidas),
    )


def _accuracy_prueba(y_real, y_predicho):
    """Calcula accuracy sin depender de la celda posterior de métricas."""
    return float(np.mean(np.asarray(y_real) == np.asarray(y_predicho)))


if "df" in globals() and "TARGET_COL" in globals():
    _X_poda = df.drop(columns=[TARGET_COL])
    _y_poda = df[TARGET_COL].to_numpy()
    _X_tr_poda, _X_te_poda, _y_tr_poda, _y_te_poda = train_test_split(
        _X_poda, _y_poda, test_size=0.2, random_state=SEED, stratify=_y_poda,
    )

    _modelos_poda = {
        "sin poda": DecisionTreeC45(pruning=False, random_state=SEED),
        "con poda": DecisionTreeC45(
            pruning=True, pruning_confidence=0.25, random_state=SEED,
        ),
        "hoja mínima grande": DecisionTreeC45(
            pruning=False, min_samples_leaf=10, random_state=SEED,
        ),
    }
    _resultados_poda = {}
    for _nombre_poda, _modelo_poda in _modelos_poda.items():
        _modelo_poda.fit(_X_tr_poda, _y_tr_poda)
        _pred_train_poda = _modelo_poda.predict(_X_tr_poda)
        _pred_test_poda = _modelo_poda.predict(_X_te_poda)
        _nodos_poda, _hojas_poda, _prof_poda = _medir_estructura_arbol(
            _modelo_poda.arbol_.raiz
        )
        _resultados_poda[_nombre_poda] = {
            "pred_test": _pred_test_poda,
            "nodos": _nodos_poda,
            "hojas": _hojas_poda,
            "profundidad": _prof_poda,
            "accuracy_train": _accuracy_prueba(_y_tr_poda, _pred_train_poda),
            "accuracy_test": _accuracy_prueba(_y_te_poda, _pred_test_poda),
        }
        _resultados_poda[_nombre_poda]["brecha"] = (
            _resultados_poda[_nombre_poda]["accuracy_train"]
            - _resultados_poda[_nombre_poda]["accuracy_test"]
        )

    _base_poda = _resultados_poda["sin poda"]
    _con_poda = _resultados_poda["con poda"]
    _hoja_grande = _resultados_poda["hoja mínima grande"]
    _coincidencia_poda = float(np.mean(
        _base_poda["pred_test"] == _con_poda["pred_test"]
    ))
    _coincidencia_hoja = float(np.mean(
        _base_poda["pred_test"] == _hoja_grande["pred_test"]
    ))

    print(
        f"Sin poda: nodos={_base_poda['nodos']}  "
        f"acc_test={_base_poda['accuracy_test']:.4f}  brecha={_base_poda['brecha']:.4f}"
    )
    print(
        f"Con poda: nodos={_con_poda['nodos']}  "
        f"acc_test={_con_poda['accuracy_test']:.4f}  brecha={_con_poda['brecha']:.4f}"
    )
    # La poda elige mirando el train. En un solo holdout la accuracy puede bajar un poco.
    # El margen de 3 puntos es el mismo que usa min_samples_leaf. La media está en la Tabla 5.
    assert _con_poda["nodos"] < _base_poda["nodos"], (
        "La poda debe reducir estrictamente la cantidad de nodos."
    )
    assert _con_poda["accuracy_test"] >= _base_poda["accuracy_test"] - 0.03, (
        "La accuracy de prueba bajó más de 3 puntos al podar "
        f"({_base_poda['accuracy_test']:.4f} → {_con_poda['accuracy_test']:.4f})."
    )
    assert _con_poda["brecha"] <= _base_poda["brecha"] + 0.03, (
        "La brecha train/test empeoró más de 3 puntos al podar "
        f"({_base_poda['brecha']:.4f} → {_con_poda['brecha']:.4f})."
    )
    assert _hoja_grande["nodos"] < _base_poda["nodos"], (
        "min_samples_leaf=10 debe producir un árbol menor en esta partición."
    )
    assert _hoja_grande["accuracy_test"] >= _base_poda["accuracy_test"] - 0.03, (
        "Con min_samples_leaf=10, la accuracy debe quedar a menos de 3 puntos "
        "porcentuales de la referencia sin poda, o ser superior."
    )
    assert _coincidencia_poda > 0.5, (
        "La poda debe conservar la mayoría de las predicciones de prueba."
    )
    assert _coincidencia_hoja > 0.5, (
        "El árbol con hoja mínima grande debe conservar la mayoría de las "
        "predicciones de prueba."
    )

    _tabla_prueba_poda = pd.DataFrame([
        {
            "configuración": _nombre,
            "nodos": _datos["nodos"],
            "hojas": _datos["hojas"],
            "profundidad máxima": _datos["profundidad"],
            "accuracy train": _datos["accuracy_train"],
            "accuracy test": _datos["accuracy_test"],
            "brecha train/test": _datos["brecha"],
        }
        for _nombre, _datos in _resultados_poda.items()
    ])
    display(_tabla_prueba_poda.round(4))
    print(
        "Coincidencia de predicciones con el árbol sin poda: "
        f"poda={_coincidencia_poda:.1%}, "
        f"min_samples_leaf=10={_coincidencia_hoja:.1%}"
    )
    print("✓ Pruebas de tamaño y efecto de la poda superadas.")
else:
    print("ℹ️ Prueba de poda: faltan df o TARGET_COL; ejecutá primero la carga de datos.")


Sin poda: nodos=448  acc_test=0.7880  brecha=0.1250
Con poda: nodos=71  acc_test=0.7989  brecha=0.0557


,configuración,nodos,hojas,profundidad máxima,accuracy train,accuracy test,brecha train/test
0,sin poda,448,244,13,0.9130,0.7880,0.1250
1,con poda,71,39,10,0.8546,0.7989,0.0557
2,hoja mínima grande,84,45,9,0.8342,0.8370,-0.0027


Coincidencia de predicciones con el árbol sin poda: poda=90.2%, min_samples_leaf=10=88.6%
✓ Pruebas de tamaño y efecto de la poda superadas.


---
# 4. Random Forest y cómo medimos

**Slot: 4º** · Cards 30–35

**Qué es Random Forest:** muchos árboles, cada uno ve una muestra/bootstrap y un subconjunto de atributos; la clase final sale por **voto**. Suele generalizar mejor que un solo árbol, a costa de ser menos interpretable.

**Por qué lo usamos:** la consigna pide comparar el DT propio con un ensamble. El PDF lo permite de forma textual: «para el algoritmo RF se podrá utilizar un modelo de la libraría Scikit-learn». Esa cita está en el informe, sección 3 (`informe/seccion3.md`).

**Interfaz compartida:** `RandomForestWrap` expone la misma API que `DecisionTreeC45`: `fit(X, y)`, `predict(X)`, `predict_proba(X)` y `score(X, y)`. `X` es un DataFrame de atributos y `y` el vector clase. El preprocesamiento de columnas mixtas queda adentro del wrapper, así el código de evaluación no distingue los dos modelos.

**Qué es el protocolo:** no alcanza un solo `train_test_split` de suerte. La constante compartida `cv` se crea en el setup (celda 1.1) y no se vuelve a instanciar:

`cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)`

Son 25 particiones estratificadas. `cross_validate` usa ese mismo objeto, con `n_jobs=-1` y `return_train_score=True`, para medir la brecha de overfitting. Cada métrica queda con 25 valores de train y 25 de test.

**Hacer:**
1. Wrapper RF con la **misma interfaz** que el DT  
2. Función única `evaluate(...)`  
3. **E1:** tabla DT vs RF  
4. **E5:** DT con vs sin poda

In [20]:
# ═══ 4. RandomForestWrap ═══
# Envuelve sklearn.ensemble.RandomForestClassifier con la misma interfaz que DecisionTreeC45:
# fit(X, y), predict(X), predict_proba(X), score(X, y).
# X es un DataFrame de atributos (sin la columna clase) e y es el vector objetivo.
# Datos mixtos: numéricos (imputación mediana) + categóricos / bool (moda + one-hot).
# Ese preprocesamiento vive acá para que el código de evaluación pase las mismas columnas crudas a DT y RF.

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split


class RandomForestWrap(ClassifierMixin, BaseEstimator):
    """Wrapper de RandomForestClassifier con la API de DecisionTreeC45.

    Parámetros: n_estimators, max_depth, max_features, min_samples_leaf,
    bootstrap y random_state (por defecto SEED).
    """

    def __init__(
        self,
        n_estimators=100,
        max_depth=None,
        max_features="sqrt",
        min_samples_leaf=1,
        bootstrap=True,
        random_state=SEED,
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_leaf = min_samples_leaf
        self.bootstrap = bootstrap
        self.random_state = random_state

    @staticmethod
    def _particionar_columnas(X):
        """Separa numéricas (no bool) de categóricas / bool / object / category."""
        num_cols = [
            c for c in X.columns
            if pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_bool_dtype(X[c])
        ]
        cat_cols = [c for c in X.columns if c not in num_cols]
        return num_cols, cat_cols

    def _armar_pipeline(self, num_cols, cat_cols):
        transformers = []
        if num_cols:
            transformers.append(("num", SimpleImputer(strategy="median"), num_cols))
        if cat_cols:
            transformers.append((
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                cat_cols,
            ))
        if not transformers:
            raise ValueError("No hay columnas de features para entrenar.")

        preprocess = ColumnTransformer(transformers, remainder="drop")
        clf = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            max_features=self.max_features,
            min_samples_leaf=self.min_samples_leaf,
            bootstrap=self.bootstrap,
            random_state=self.random_state,
        )
        return Pipeline([("preprocess", preprocess), ("rf", clf)])

    def _X(self, X):
        if not hasattr(self, "pipeline_"):
            raise RuntimeError("El modelo no fue entrenado todavía (falta fit).")
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X debe ser un DataFrame.")
        faltantes = [c for c in self.feature_cols_ if c not in X.columns]
        if faltantes:
            raise KeyError(f"Faltan columnas de features vistas en fit: {faltantes}")
        return X.loc[:, self.feature_cols_]

    def fit(self, X, y):
        """Entrena con las mismas entradas que DecisionTreeC45.fit(X, y)."""
        if not isinstance(X, pd.DataFrame):
            raise TypeError("X debe ser un DataFrame.")
        y = np.asarray(y)
        if len(X) != len(y):
            raise ValueError("X e y deben tener la misma cantidad de filas.")
        if len(X) == 0:
            raise ValueError("No se puede entrenar con un DataFrame vacío.")
        X = X.reset_index(drop=True)
        self.feature_cols_ = list(X.columns)
        self.num_cols_, self.cat_cols_ = self._particionar_columnas(X)
        self.pipeline_ = self._armar_pipeline(self.num_cols_, self.cat_cols_)
        self.pipeline_.fit(X, y)
        self.classes_ = np.asarray(self.pipeline_.named_steps["rf"].classes_, dtype=object)
        return self

    def predict(self, X):
        return self.pipeline_.predict(self._X(X))

    def predict_proba(self, X):
        return self.pipeline_.predict_proba(self._X(X))

    def score(self, X, y, sample_weight=None):
        """Accuracy. Firma idéntica a ClassifierMixin.score, la que usa DecisionTreeC45."""
        return float(accuracy_score(y, self.predict(X), sample_weight=sample_weight))


def _separar_xy(datos, target):
    """Arma el par (X, y) que comparten DT y RF."""
    if target not in datos.columns:
        raise KeyError(f"target='{target}' no está en el DataFrame.")
    return datos.drop(columns=[target]), datos[target]


# El modelo se entrena solo con el dataset cargado (df, TARGET_COL).
if "df" in dir() and "TARGET_COL" in dir():
    _X_real, _y_real = _separar_xy(df, TARGET_COL)
    _X_train, _X_test, _y_train, _y_test = train_test_split(
        _X_real, _y_real, test_size=0.2, random_state=SEED, stratify=_y_real
    )
    _rf_real = RandomForestWrap(
        n_estimators=100,
        max_depth=None,
        max_features="sqrt",
        min_samples_leaf=1,
        bootstrap=True,
        random_state=SEED,
    )
    _rf_real.fit(_X_train, _y_train)
    _acc_real = _rf_real.score(_X_test, _y_test)
    _proba_real = _rf_real.predict_proba(_X_test)
    assert _proba_real.shape == (len(_X_test), len(_rf_real.classes_))
    print(f"\n✓ RandomForestWrap con dataset real ({NOMBRE_DATASET})")
    print(f"  Shape train/test : {len(_X_train)} / {len(_X_test)}")
    print(f"  num_cols         : {len(_rf_real.num_cols_)} | cat_cols: {len(_rf_real.cat_cols_)}")
    print(f"  Accuracy holdout : {_acc_real:.4f}")
    print(f"  predict_proba    : shape {_proba_real.shape}")
else:
    print("\nℹ️  Dataset real no cargado todavía (correr celdas 1.1 y 1.2 antes).")



✓ RandomForestWrap con dataset real (heart disease (Cleveland, Hungría, Suiza, Long Beach))
  Shape train/test : 736 / 184
  num_cols         : 5 | cat_cols: 8
  Accuracy holdout : 0.8641
  predict_proba    : shape (184, 2)


In [21]:
# ═══ 4. Métricas (+ evaluate pendiente) ═══
# metricas(...) arma un dict listo para DataFrame.
# Se reportan macro Y weighted (no son equivalentes con desbalance).
# COMPARACIÓN DT vs RF: la métrica principal es f1_macro (cada clase pesa igual).
#   Weighted y accuracy se publican en la misma tabla; no deciden el ganador.

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
import warnings

# Clave de metricas() que decide la comparación DT vs RF.
# Las demás se reportan; no eligen al ganador.
METRICA_COMPARACION = "f1_macro"


def metricas(y_true, y_pred, y_proba=None, clases=None):
    """Dict de métricas de clasificación (binario y multiclase).

    Parámetros
    ----------
    y_true, y_pred : array-like
        Etiquetas reales y predichas.
    y_proba : array-like o None
        Probabilidades (n_muestras, n_clases), alineadas a `clases`.
        Si se pasa, se agrega ROC-AUC (OVR, average macro/weighted).
    clases : array-like o None
        Orden/etiquetas de clase. Si None, sklearn las infiere.

    Returns
    -------
    dict[str, float]
        Claves listas para una fila de DataFrame. Sin NaN.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    labels = None if clases is None else list(clases)

    kwargs = {"labels": labels, "zero_division": 0}

    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(
            precision_score(y_true, y_pred, average="macro", **kwargs)
        ),
        "precision_weighted": float(
            precision_score(y_true, y_pred, average="weighted", **kwargs)
        ),
        "recall_macro": float(
            recall_score(y_true, y_pred, average="macro", **kwargs)
        ),
        "recall_weighted": float(
            recall_score(y_true, y_pred, average="weighted", **kwargs)
        ),
        "f1_macro": float(
            f1_score(y_true, y_pred, average="macro", **kwargs)
        ),
        "f1_weighted": float(
            f1_score(y_true, y_pred, average="weighted", **kwargs)
        ),
    }

    # ROC-AUC opcional (sirve cuando hay predict_proba)
    if y_proba is not None:
        P = np.asarray(y_proba, dtype=float)
        clases_auc = labels if labels is not None else list(np.unique(y_true))
        try:
            if P.ndim != 2 or P.shape[0] != len(y_true):
                raise ValueError(
                    f"y_proba debe ser (n_muestras, n_clases); llegó {P.shape}"
                )
            # Binario: sklearn acepta vector de proba de la clase positiva
            if P.shape[1] == 2:
                score_vec = P[:, 1]
                out["roc_auc"] = float(
                    roc_auc_score(y_true, score_vec, labels=clases_auc)
                )
            else:
                out["roc_auc_macro"] = float(
                    roc_auc_score(
                        y_true, P, multi_class="ovr", average="macro",
                        labels=clases_auc,
                    )
                )
                out["roc_auc_weighted"] = float(
                    roc_auc_score(
                        y_true, P, multi_class="ovr", average="weighted",
                        labels=clases_auc,
                    )
                )
        except ValueError as err:
            # p.ej. una sola clase en y_true → no se puede AUC; dejamos None tipado como nan evitado
            warnings.warn(f"ROC-AUC omitido: {err}", stacklevel=2)

    # Garantía: sin NaN en los valores numéricos reportados
    for k, v in list(out.items()):
        if v is None or (isinstance(v, float) and (np.isnan(v) or np.isinf(v))):
            out[k] = 0.0

    if METRICA_COMPARACION not in out:
        raise KeyError(f"{METRICA_COMPARACION} no está en el dict de métricas.")
    return out


# ═══ 4. Protocolo CV compartido (card 33) ═══
# cv vive en la celda 1.1. Esta celda solo lo usa.
# Dos procesos como máximo. Con ocho, la máquina se queda sin memoria.
# return_train_score=True deja la brecha train−test.

from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, roc_auc_score


def _roc_auc(estimador, X, y):
    """ROC-AUC macro. Sirve en binario y en multiclase."""
    proba = estimador.predict_proba(X)
    y = np.asarray(y)
    clases = list(getattr(estimador, "classes_", []))
    if proba.shape[1] == 2:
        return float(roc_auc_score(y, proba[:, 1]))
    return float(roc_auc_score(y, proba, multi_class="ovr", average="macro", labels=clases))


# Mismas métricas que metricas(), para que la tabla de CV y el resto del TP midan lo mismo.
SCORING = {
    "accuracy": "accuracy",
    "precision_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "precision_weighted": make_scorer(precision_score, average="weighted", zero_division=0),
    "recall_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "recall_weighted": make_scorer(recall_score, average="weighted", zero_division=0),
    "f1_macro": make_scorer(f1_score, average="macro", zero_division=0),
    "f1_weighted": make_scorer(f1_score, average="weighted", zero_division=0),
    "roc_auc": _roc_auc,
}


def tabla_cv(resultado, nombre):
    """Una fila por partición y métrica. 5×5 = 25 filas de cada métrica."""
    nombres = [clave[len("test_"):] for clave in resultado if clave.startswith("test_")]
    filas = []
    for i in range(len(resultado["fit_time"])):
        n_splits = cv.get_n_splits() // cv.n_repeats
        repeat, fold = divmod(i, n_splits)
        for metrica in nombres:
            train = float(resultado[f"train_{metrica}"][i])
            test = float(resultado[f"test_{metrica}"][i])
            filas.append({
                "modelo": nombre,
                "repeat": repeat,
                "fold": fold,
                "metrica": metrica,
                "train": train,
                "test": test,
                "brecha": train - test,
            })
    return pd.DataFrame(filas)


def _trabajos_cv():
    """Dos procesos como máximo. El cv no cambia."""
    import os
    nucleos = os.cpu_count() or 1
    return 2 if nucleos >= 2 else 1


def evaluar_cv(modelo, X, y):
    """cross_validate contra el cv del setup. No construye otro splitter."""
    if "cv" not in globals() or not isinstance(cv, RepeatedStratifiedKFold):
        raise RuntimeError("Falta el cv compartido. Correr la celda 1.1 antes.")
    if cv.random_state != SEED or cv.n_repeats != 5 or cv.get_n_splits() != 25:
        raise RuntimeError("cv fue reemplazado. Tiene que ser el del setup, con SEED.")
    resultado = cross_validate(
        modelo,
        X,
        y,
        cv=cv,
        scoring=SCORING,
        n_jobs=_trabajos_cv(),
        return_train_score=True,
        error_score="raise",
    )
    return tabla_cv(resultado, type(modelo).__name__)


def _hojas_y_altura(nodo, profundidad=0):
    """Hojas del subárbol y altura (aristas hasta la hoja más lejana)."""
    if getattr(nodo, "es_hoja", False) or not nodo.hijos:
        return 1, profundidad
    hojas = 0
    altura = profundidad
    for hijo in nodo.hijos.values():
        sub_hojas, sub_altura = _hojas_y_altura(hijo, profundidad + 1)
        hojas += sub_hojas
        altura = max(altura, sub_altura)
    return hojas, altura


def tamano_modelo(estimador):
    """Nodos, hojas y altura. El DT se recorre; el RF suma los árboles de sklearn."""
    if hasattr(estimador, "arbol_"):
        raiz = estimador.arbol_.raiz
        hojas, altura = _hojas_y_altura(raiz)
        return {
            "nodos": int(contar_nodos_arbol(raiz)),
            "hojas": int(hojas),
            "altura": int(altura),
        }
    bosque = estimador.pipeline_.named_steps["rf"].estimators_
    return {
        "nodos": int(sum(arbol.tree_.node_count for arbol in bosque)),
        "hojas": int(sum(arbol.tree_.n_leaves for arbol in bosque)),
        "altura": int(max(arbol.tree_.max_depth for arbol in bosque)),
    }


def construir_tabla2(modelos, X, y, ya_medido=None):
    """Tabla 2 y Fig. 4. Si ya_medido trae la pasada, no vuelve a ajustar."""
    if "cv" not in globals() or not isinstance(cv, RepeatedStratifiedKFold):
        raise RuntimeError("Falta el cv compartido. Correr la celda 1.1 antes.")
    import matplotlib.pyplot as plt

    largo = []
    crudo = []
    for nombre, modelo in modelos.items():
        if ya_medido and nombre in ya_medido:
            print(f"  E1 reutiliza la pasada de {nombre}", flush=True)
            resultado = ya_medido[nombre]
        else:
            print(f"  E1 ajustando {nombre} (25 folds)", flush=True)
            resultado = cross_validate(
                modelo,
                X,
                y,
                cv=cv,
                scoring=SCORING,
                n_jobs=_trabajos_cv(),
                return_train_score=True,
                return_estimator=True,
                error_score="raise",
            )
        tabla = tabla_cv(resultado, nombre)
        conteo = tabla.groupby("metrica").size()
        if not (conteo == 25).all():
            raise RuntimeError(f"{nombre}: se esperaban 25 filas por métrica, llegó {conteo.to_dict()}.")
        for metrica, grupo in tabla.groupby("metrica"):
            largo.append({
                "metrica": metrica,
                "modelo": nombre,
                "media": float(grupo["test"].mean()),
                "desvio": float(grupo["test"].std(ddof=1)),
            })
        for i, estimador in enumerate(resultado["estimator"]):
            tam = tamano_modelo(estimador)
            crudo.append({
                "modelo": nombre,
                "tiempo_fit": float(resultado["fit_time"][i]),
                "tiempo_predict": float(resultado["score_time"][i]),
                **tam,
            })
        del resultado

    extras = pd.DataFrame(crudo)
    for columna in ("tiempo_fit", "tiempo_predict", "nodos", "hojas", "altura"):
        for modelo, grupo in extras.groupby("modelo"):
            largo.append({
                "metrica": columna,
                "modelo": modelo,
                "media": float(grupo[columna].mean()),
                "desvio": float(grupo[columna].std(ddof=1)),
            })
    largo = pd.DataFrame(largo)
    orden = list(SCORING) + ["tiempo_fit", "tiempo_predict", "nodos", "hojas", "altura"]
    filas = []
    for metrica in orden:
        fila = {"metrica": metrica}
        for modelo in modelos:
            sub = largo[(largo["metrica"] == metrica) & (largo["modelo"] == modelo)]
            if len(sub) != 1:
                raise RuntimeError(f"Falta {metrica} para {modelo}.")
            fila[f"{modelo} media"] = float(sub["media"].iloc[0])
            fila[f"{modelo} desvio"] = float(sub["desvio"].iloc[0])
        filas.append(fila)
    tabla2 = pd.DataFrame(filas)

    DIR_FIGURAS.mkdir(exist_ok=True)
    ruta_tex = DIR_FIGURAS / "tabla2_dt_vs_rf.tex"
    tabla2.to_latex(
        ruta_tex,
        index=False,
        float_format="%.4f",
        caption="Tabla 2. DT vs RF: media y desvío en test, tiempos (s) y tamaño.",
        label="tab:dt-vs-rf",
    )

    metricas_fig = list(SCORING)
    fig, ax = plt.subplots(figsize=(10.5, 4.8))
    x = np.arange(len(metricas_fig))
    ancho = 0.36
    nombres = list(modelos)
    for desplazamiento, modelo in ((-ancho / 2, nombres[0]), (ancho / 2, nombres[1])):
        medias = [
            float(tabla2.loc[tabla2["metrica"] == m, f"{modelo} media"].iloc[0])
            for m in metricas_fig
        ]
        desvios = [
            float(tabla2.loc[tabla2["metrica"] == m, f"{modelo} desvio"].iloc[0])
            for m in metricas_fig
        ]
        ax.bar(x + desplazamiento, medias, ancho, yerr=desvios, capsize=3, label=modelo)
    ax.set_xticks(x)
    ax.set_xticklabels(metricas_fig, rotation=35, ha="right")
    ax.set_ylabel("Media en test")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Fig. 4  DT vs RF  (comparación: {METRICA_COMPARACION})")
    ax.legend()
    fig.tight_layout()
    ruta_fig = DIR_FIGURAS / "fig4_dt_vs_rf.png"
    fig.savefig(ruta_fig, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return tabla2, ruta_tex, ruta_fig


def _folds_conservan_clases(X, y):
    """Cada fold de test tiene que incluir todas las clases del dataset."""
    y_arr = np.asarray(y)
    clases = set(np.unique(y_arr))
    for _, test in cv.split(X, y_arr):
        if set(np.unique(y_arr[test])) != clases:
            return False
    return True


if "df" in dir() and "TARGET_COL" in dir() and "DecisionTreeC45" in dir() and "RandomForestWrap" in dir():
    _X_cv, _y_cv = _separar_xy(df, TARGET_COL)
    assert cv.get_n_splits() == 25
    assert _folds_conservan_clases(_X_cv, _y_cv), "Un fold perdió una clase: el protocolo no está estratificado."

    _modelos_cv = {
        "DT": DecisionTreeC45(random_state=SEED),
        "RF": RandomForestWrap(
            n_estimators=100,
            max_depth=None,
            max_features="sqrt",
            min_samples_leaf=1,
            bootstrap=True,
            random_state=SEED,
        ),
    }
    _trabajos = _trabajos_cv()
    print(
        f"Una pasada, 25 particiones, {_trabajos} procesos. "
        "El árbol tarda cerca de un minuto por ajuste: con 2 procesos, unos 15 minutos. "
        "El bosque termina enseguida. La semilla fija la corrida, no hace falta repetirla.",
        flush=True,
    )
    _medidos = {}
    _tablas_cv = []
    for _nombre, _modelo in _modelos_cv.items():
        _clase = type(_modelo).__name__
        print(f"Pasada {_clase}.", flush=True)
        _resultado = cross_validate(
            _modelo, _X_cv, _y_cv, cv=cv, scoring=SCORING,
            n_jobs=_trabajos, return_train_score=True,
            return_estimator=True, error_score="raise",
        )
        _primera = tabla_cv(_resultado, _clase)
        _medidos[_clase] = _resultado
        _por_metrica = _primera.groupby("metrica").size()
        assert (_por_metrica == 25).all(), _por_metrica.to_dict()
        _tablas_cv.append(_primera)
        print(f"✓ {_clase}: 25 filas por métrica", flush=True)

    tabla_cv_modelos = pd.concat(_tablas_cv, ignore_index=True)
    resumen_cv = (
        tabla_cv_modelos.groupby(["modelo", "metrica"])[["train", "test", "brecha"]]
        .agg(["mean", "std"])
    )
    print("\n✓ cross_validate DT y RF")
    display(resumen_cv.round(4))
    tabla2, _ruta_tex, _ruta_fig = construir_tabla2(
        {
            "DecisionTreeC45": DecisionTreeC45(random_state=SEED),
            "RandomForestWrap": RandomForestWrap(
                n_estimators=100,
                max_depth=None,
                max_features="sqrt",
                min_samples_leaf=1,
                bootstrap=True,
                random_state=SEED,
            ),
        },
        _X_cv,
        _y_cv,
        ya_medido=_medidos,
    )
    print(f"\n✓ Tabla 2  {_ruta_tex}")
    print(f"✓ Fig. 4   {_ruta_fig}")
    display(tabla2.round(4))
else:
    print("\nℹ️  CV: faltan df, DecisionTreeC45 o RandomForestWrap. Correr las celdas anteriores.")

print("ℹ️  E5 (DT con poda vs sin poda) sigue pendiente.")


CV en 4 procesos.


✓ DT: 25 filas por métrica y dos corridas idénticas


✓ RF: 25 filas por métrica y dos corridas idénticas

✓ cross_validate DT y RF


train            test          brecha  \
                                       mean     std    mean     std    mean   
modelo           metrica                                                      
DecisionTreeC45  accuracy            0.9285  0.0086  0.7850  0.0265  0.1435   
                 f1_macro            0.9273  0.0088  0.7809  0.0271  0.1465   
                 f1_weighted         0.9283  0.0086  0.7840  0.0266  0.1443   
                 precision_macro     0.9305  0.0089  0.7848  0.0275  0.1457   
                 precision_weighted  0.9294  0.0086  0.7856  0.0269  0.1438   
                 recall_macro        0.9254  0.0089  0.7796  0.0270  0.1458   
                 recall_weighted     0.9285  0.0086  0.7850  0.0265  0.1435   
                 roc_auc             0.9821  0.0041  0.8195  0.0295  0.1626   
RandomForestWrap accuracy            1.0000  0.0000  0.8187  0.0305  0.1813   
                 f1_macro            1.0000  0.0000  0.8157  0.0308  0.1843   
                 f1_weighted         1.0000  0.0000  0.8182  0.0304  0.1818   
                 precision_macro     1.0000  0.0000  0.8185  0.0317  0.1815   
                 precision_weighted  1.0000  0.0000  0.8192  0.0307  0.1808   
                 recall_macro        1.0000  0.0000  0.8145  0.0303  0.1855   
                 recall_weighted     1.0000  0.0000  0.8187  0.0305  0.1813   
                 roc_auc             1.0000  0.0000  0.8839  0.0254  0.1161   

                                             
                                        std  
modelo           metrica                     
DecisionTreeC45  accuracy            0.0310  
                 f1_macro            0.0317  
                 f1_weighted         0.0312  
                 precision_macro     0.0323  
                 precision_weighted  0.0316  
                 recall_macro        0.0316  
                 recall_weighted     0.0310  
                 roc_auc             0.0319  
RandomForestWrap accuracy            0.0305  
                 f1_macro            0.0308  
                 f1_weighted         0.0304  
                 precision_macro     0.0317  
                 precision_weighted  0.0307  
                 recall_macro        0.0303  
                 recall_weighted     0.0305  
                 roc_auc             0.0254

  E1 ajustando DecisionTreeC45 (25 folds)


  E1 ajustando RandomForestWrap (25 folds)



✓ Tabla 2  figuras\tabla2_dt_vs_rf.tex
✓ Fig. 4   figuras\fig4_dt_vs_rf.png


,metrica,DecisionTreeC45 media,DecisionTreeC45 desvio,RandomForestWrap media,RandomForestWrap desvio
0,accuracy,0.7850,0.0265,0.8187,0.0305
1,precision_macro,0.7848,0.0275,0.8185,0.0317
2,precision_weighted,0.7856,0.0269,0.8192,0.0307
3,recall_macro,0.7796,0.0270,0.8145,0.0303
4,recall_weighted,0.7850,0.0265,0.8187,0.0305
5,f1_macro,0.7809,0.0271,0.8157,0.0308
6,f1_weighted,0.7840,0.0266,0.8182,0.0304
7,roc_auc,0.8195,0.0295,0.8839,0.0254
8,tiempo_fit,50.5863,9.7970,0.4842,0.0406
9,tiempo_predict,0.1510,0.0324,0.1355,0.0228


ℹ️  E5 (DT con poda vs sin poda) sigue pendiente.


### Tabla 2 — DT vs RF

Media ± desvío en las 25 particiones del `cv` compartido, sobre Heart Disease (920 filas, cuatro sedes). La comparación que decide es `f1_macro` (`METRICA_COMPARACION`): cada clase pesa igual. Accuracy y weighted se publican y no eligen al ganador.

El tamaño del árbol se recorre sobre `arbol_.raiz`. El del bosque suma nodos y hojas de sus 100 árboles; la altura es la del más profundo. Los tiempos salen de `fit_time` y `score_time`.

| | DT | RF |
|---|---|---|
| accuracy | 0,7850 ± 0,0265 | 0,8187 ± 0,0305 |
| precision_macro | 0,7848 ± 0,0275 | 0,8185 ± 0,0317 |
| precision_weighted | 0,7856 ± 0,0269 | 0,8192 ± 0,0307 |
| recall_macro | 0,7796 ± 0,0270 | 0,8145 ± 0,0303 |
| recall_weighted | 0,7850 ± 0,0265 | 0,8187 ± 0,0305 |
| f1_macro | 0,7809 ± 0,0271 | 0,8157 ± 0,0308 |
| f1_weighted | 0,7840 ± 0,0266 | 0,8182 ± 0,0304 |
| roc_auc | 0,8195 ± 0,0295 | 0,8839 ± 0,0254 |
| tiempo_fit (s) | 50,5863 ± 9,7970 | 0,4842 ± 0,0406 |
| tiempo_predict (s) | 0,1510 ± 0,0324 | 0,1355 ± 0,0228 |
| nodos | 452,00 ± 36,44 | 28876,48 ± 664,66 |
| hojas | 247,36 ± 18,63 | 14488,24 ± 332,33 |
| altura | 13,00 ± 0,00 | 23,04 ± 1,31 |

En `f1_macro` el bosque queda arriba (0,8157 frente a 0,7809). El árbol tiene 452 nodos y altura 13. El bosque suma unos 28876 nodos. El ajuste tarda unos 50,6 s por partición en el árbol y 0,48 s en el bosque. La Fig. 4 está en `figuras/fig4_dt_vs_rf.png` y la tabla en `figuras/tabla2_dt_vs_rf.tex`.


In [22]:
# 4. E5 — Tabla 5, DT con poda y sin poda
# Misma grilla del cv compartido. Si el árbol podado tuviera más nodos, no se publica.

def construir_tabla5(X, y):
    """Tabla 5: media y desvío de tamaño, accuracy y brecha, con poda y sin poda."""
    if "cv" not in globals() or not isinstance(cv, RepeatedStratifiedKFold):
        raise RuntimeError("Falta el cv compartido. Correr la celda 1.1 antes.")
    variantes = (
        ("sin_poda", DecisionTreeC45(pruning=False, random_state=SEED)),
        ("con_poda", DecisionTreeC45(pruning=True, pruning_confidence=0.25, random_state=SEED)),
    )
    crudo = []
    for nombre, modelo in variantes:
        print(f"  E5 ajustando {nombre} (25 folds)", flush=True)
        resultado = cross_validate(
            modelo, X, y, cv=cv, scoring="accuracy",
            n_jobs=_trabajos_cv(), return_train_score=True, return_estimator=True, error_score="raise",
        )
        if len(resultado["estimator"]) != 25:
            raise RuntimeError(f"{nombre}: se esperaban 25 árboles.")
        for i, estimador in enumerate(resultado["estimator"]):
            tam = tamano_modelo(estimador)
            train = float(resultado["train_score"][i])
            test = float(resultado["test_score"][i])
            crudo.append({
                "variante": nombre,
                "nodos": tam["nodos"],
                "hojas": tam["hojas"],
                "altura": tam["altura"],
                "accuracy_train": train,
                "accuracy_test": test,
                "brecha": train - test,
            })
        del resultado
    largo = pd.DataFrame(crudo)
    orden = ["nodos", "hojas", "altura", "accuracy_train", "accuracy_test", "brecha"]
    filas = []
    for metrica in orden:
        fila = {"metrica": metrica}
        for nombre, _ in variantes:
            serie = largo.loc[largo["variante"] == nombre, metrica]
            if len(serie) != 25:
                raise RuntimeError(f"Faltan filas de {nombre} en {metrica}.")
            fila[f"{nombre} media"] = float(serie.mean())
            fila[f"{nombre} desvio"] = float(serie.std(ddof=1))
        filas.append(fila)
    tabla5 = pd.DataFrame(filas)
    nodos = tabla5.set_index("metrica")
    if float(nodos.loc["nodos", "con_poda media"]) > float(nodos.loc["nodos", "sin_poda media"]):
        raise RuntimeError(
            "El árbol con poda tiene más nodos que el que crece sin poda. "
            "Revisar podar_pesimista antes de publicar la Tabla 5."
        )
    DIR_FIGURAS.mkdir(exist_ok=True)
    ruta_tex = DIR_FIGURAS / "tabla5_poda.tex"
    tabla5.to_latex(
        ruta_tex, index=False, float_format="%.4f",
        caption="Tabla 5. DT con poda pesimista y sin poda: media y desvío en 25 particiones.",
        label="tab:poda",
    )
    return tabla5, ruta_tex


if "df" in dir() and "TARGET_COL" in dir() and "DecisionTreeC45" in dir() and "tamano_modelo" in dir():
    _X_e5, _y_e5 = _separar_xy(df, TARGET_COL)
    tabla5, _ruta_tex5 = construir_tabla5(_X_e5, _y_e5)
    print(f"✓ Tabla 5  {_ruta_tex5}")
    display(tabla5.round(4))
else:
    print("ℹ️  Tabla 5: faltan df, DecisionTreeC45 o tamano_modelo.")


  E5 ajustando sin_poda (25 folds)


  E5 ajustando con_poda (25 folds)


✓ Tabla 5  figuras\tabla5_poda.tex


,metrica,sin_poda media,sin_poda desvio,con_poda media,con_poda desvio
0,nodos,452.0000,36.4406,71.2000,15.7374
1,hojas,247.3600,18.6299,39.0400,8.4335
2,altura,13.0000,0.0000,8.8400,0.8981
3,accuracy_train,0.9285,0.0086,0.8663,0.0151
4,accuracy_test,0.7850,0.0265,0.7741,0.0284
5,brecha,0.1435,0.0310,0.0922,0.0353


### Tabla 5 — DT con poda y sin poda

La celda de arriba compara `DecisionTreeC45(pruning=False)` con `DecisionTreeC45(pruning=True, pruning_confidence=0.25)` en las mismas 25 particiones. Cada celda es media ± desvío. Sin poda coincide con el DT de la Tabla 2: 452 nodos y accuracy de prueba 0,7850.

| | sin poda | con poda |
|---|---|---|
| nodos | 452,00 ± 36,44 | 71,20 ± 15,74 |
| hojas | 247,36 ± 18,63 | 39,04 ± 8,43 |
| altura | 13,00 ± 0,00 | 8,84 ± 0,90 |
| accuracy train | 0,9285 ± 0,0086 | 0,8663 ± 0,0151 |
| accuracy test | 0,7850 ± 0,0265 | 0,7741 ± 0,0284 |
| brecha | 0,1435 ± 0,0310 | 0,0922 ± 0,0353 |

La poda achica el árbol (452 nodos contra 71). La accuracy de prueba baja (0,7850 a 0,7741) y la brecha también (0,1435 a 0,0922). Sin poda el train queda en 0,9285, no en 1. El LaTeX queda en `figuras/tabla5_poda.tex`.


### Fig. 9 — Matrices de confusión y curvas ROC

**Card 32.** Los dos modelos se evalúan sobre la misma partición estratificada 80/20 (`random_state=SEED`). No hay nombres de clase escritos en el código: salen de `classes_` del modelo, en el mismo orden que las columnas de `predict_proba`.

Con dos clases, la segunda columna es la positiva (convención de scikit-learn, no una clase elegida a mano). La matriz queda:

| | Predicho negativo | Predicho positivo |
|---|---|---|
| Verdadero negativo | TN | FP |
| Verdadero positivo | FN | TP |

Con tres o más clases no hay una sola clase positiva. La Fig. 9 usa los nombres reales del dataset. TN, FP, FN y TP se calculan una vez por clase, contra el resto, y se guardan en `figuras/tabla_tn_fp_fn_tp.csv`.

ROC-AUC: con tres o más clases, `roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")`. Con dos clases scikit-learn rechaza esa firma sobre una matriz de dos columnas; el macro es el promedio de las curvas one-vs-rest. La Fig. 10 dibuja una curva por clase, sea cual sea el nombre.

Archivos fijos: `figuras/fig9_matrices_confusion.png` y `figuras/fig10_roc_por_clase.png`.


En esta corrida el holdout estratificado 80/20, con `random_state=SEED`, tiene 736 filas de entrenamiento y 184 de prueba. La segunda clase de `classes_` es la positiva: presencia.

| Modelo | TN | FP | FN | TP |
|---|---|---|---|---|
| DecisionTreeC45 | 57 | 25 | 14 | 88 |
| RandomForestWrap | 67 | 15 | 10 | 92 |

El ROC-AUC macro de ese holdout es 0,8192 para el árbol y 0,9219 para el bosque. Es un solo corte, no la media de las 25 particiones de la Tabla 2.

En el test hay 82 ausencia y 102 presencia. Cada fila de la matriz está normalizada.

| | Error en ausencia | Error en presencia |
|---|---|---|
| DecisionTreeC45 | 30,5% (25 de 82) | 13,7% (14 de 102) |
| RandomForestWrap | 18,3% (15 de 82) | 9,8% (10 de 102) |


In [23]:
# 4. Fig. 9 y Fig. 10 — confusión y ROC-AUC
# Mismo holdout para DT y RF. El orden de la matriz es el de las columnas de predict_proba.

import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    auc,
)
from sklearn.model_selection import train_test_split


def roc_auc_ovr_macro(y_true, proba, orden):
    """ROC-AUC macro one-vs-rest, calculado con la matriz de predict_proba.

    Con 3 o más clases llama a
    roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro").
    Con 2 clases esa firma falla: sklearn espera el vector de la clase positiva,
    no una matriz de dos columnas. El promedio de las curvas por clase es el macro OVR
    y, en binario, coincide con el AUC de la segunda columna.
    """
    y_true = np.asarray(y_true)
    orden = list(orden)
    if np.asarray(proba).shape[1] != len(orden):
        raise ValueError("Las columnas de predict_proba no coinciden con el orden de clases.")
    if len(orden) > 2:
        return float(roc_auc_score(
            y_true, proba, multi_class="ovr", average="macro", labels=orden,
        ))
    por_clase = [
        float(roc_auc_score((y_true == clase).astype(int), proba[:, i]))
        for i, clase in enumerate(orden)
    ]
    return float(np.mean(por_clase))


def proba_en_orden(modelo, X, orden):
    """Columnas de predict_proba alineadas a `orden`."""
    proba = np.asarray(modelo.predict_proba(X), dtype=float)
    clases = list(modelo.classes_)
    if clases != list(orden):
        proba = proba[:, [clases.index(c) for c in orden]]
    return proba



def tabla_tn_fp_fn_tp(y_true, y_pred, orden):
    """TP, FP, TN y FN tomando cada clase como positiva y al resto como negativa.

    Sirve con 2 o más clases. Los nombres salen de `orden`, que es `classes_`.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    filas = []
    for clase in orden:
        real = y_true == clase
        predicho = y_pred == clase
        filas.append({
            "clase_positiva": clase,
            "TP": int(np.sum(real & predicho)),
            "FP": int(np.sum(~real & predicho)),
            "TN": int(np.sum(~real & ~predicho)),
            "FN": int(np.sum(real & ~predicho)),
        })
    return pd.DataFrame(filas)


def figura_confusion_roc(modelos, X_test, y_test, orden, destino=None, clase_positiva=None):
    """Guarda la Fig. 9 (matrices lado a lado) y la Fig. 10 (ROC por clase).

    `orden` son las clases del modelo. `clase_positiva`, si se pasa, se mueve
    al final: en binario esa queda como positiva (TN/FP/FN/TP). Si no se pasa,
    se usa el orden de `classes_`, y la positiva es la segunda columna.
    """
    if destino is None:
        destino = DIR_FIGURAS
    destino.mkdir(exist_ok=True)
    orden = list(orden)
    if clase_positiva is not None:
        if clase_positiva not in orden:
            raise ValueError(f"clase_positiva={clase_positiva!r} no está en {orden}.")
        orden = [c for c in orden if c != clase_positiva] + [clase_positiva]
    y_test = np.asarray(y_test)
    n = len(modelos)

    fig, axes = plt.subplots(1, n, figsize=(6.4 * n, 5.0))
    axes = np.atleast_1d(axes)
    aucs = {}
    curvas = {}
    tablas = []
    for ax, (nombre, modelo) in zip(axes, modelos.items()):
        proba = proba_en_orden(modelo, X_test, orden)
        pred = modelo.predict(X_test)
        cm = confusion_matrix(y_test, pred, labels=orden)
        if cm.shape != (len(orden), len(orden)):
            raise ValueError(f"Matriz {cm.shape} distinta del orden de {len(orden)} clases.")
        # Los nombres de los ejes son las clases del dataset, en el orden de predict_proba.
        # Con 2 clases, la segunda es la positiva y las celdas se leen TN FP / FN TP.
        # Con más clases, la matriz queda clase×clase; TN, FP, FN y TP van a la tabla, una fila por clase.
        tabla = tabla_tn_fp_fn_tp(y_test, pred, orden)
        tabla.insert(0, "modelo", nombre)
        tablas.append(tabla)
        if len(orden) == 2:
            etiquetas = [f"{orden[0]}\n(negativa)", f"{orden[1]}\n(positiva)"]
            positiva = tabla.set_index("clase_positiva").loc[orden[1]]
            if not (
                cm[0, 0] == positiva["TN"] and cm[0, 1] == positiva["FP"]
                and cm[1, 0] == positiva["FN"] and cm[1, 1] == positiva["TP"]
            ):
                raise ValueError(f"{nombre}: TN/FP/FN/TP no coinciden con el orden de las columnas.")
            nombres_celda = ["TN", "FP", "FN", "TP"]
            valores_celda = [int(cm[0, 0]), int(cm[0, 1]), int(cm[1, 0]), int(cm[1, 1])]
        else:
            etiquetas = [str(c) for c in orden]
            nombres_celda = None
            valores_celda = None
        # Binario: cada fila suma 1, así se lee el porcentaje de error de esa clase.
        # Con más de dos clases se dejan los conteos. from_estimator usa el mismo predict que el resto de la figura.
        normalizar = "true" if len(orden) == 2 else None
        disp = ConfusionMatrixDisplay.from_estimator(
            modelo,
            X_test,
            y_test,
            labels=orden,
            display_labels=etiquetas,
            normalize=normalizar,
            ax=ax,
            colorbar=False,
        )
        if nombres_celda is not None:
            totales = cm.sum(axis=1, keepdims=True)
            proporciones = np.divide(
                cm, totales, out=np.zeros_like(cm, dtype=float), where=totales > 0,
            )
            for texto, etiqueta, valor, proporcion in zip(
                disp.text_.ravel(), nombres_celda, valores_celda, proporciones.ravel(),
            ):
                texto.set_text(f"{etiqueta}\n{proporcion:.1%}\n{valor}")
        ticks_x = [t.get_text() for t in ax.get_xticklabels()]
        ticks_y = [t.get_text() for t in ax.get_yticklabels()]
        if ticks_x != etiquetas or ticks_y != etiquetas:
            raise ValueError(f"{nombre}: los ejes de la matriz no siguen el orden de las columnas.")
        ax.set_xlabel("Predicción")
        ax.set_ylabel("Valor real")
        ax.set_title(nombre)
        aucs[nombre] = roc_auc_ovr_macro(y_test, proba, orden)
        curvas[nombre] = {}
        for i, clase in enumerate(orden):
            fpr, tpr, _ = roc_curve((y_test == clase).astype(int), proba[:, i])
            curvas[nombre][clase] = (fpr, tpr, float(auc(fpr, tpr)))
    fig.tight_layout()
    ruta_cm = destino / "fig9_matrices_confusion.png"
    fig.savefig(ruta_cm, dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 4.2), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (nombre, _) in zip(axes, modelos.items()):
        for clase, (fpr, tpr, area) in curvas[nombre].items():
            ax.plot(fpr, tpr, label=f"{clase}  {area:.3f}")
        ax.plot([0, 1], [0, 1], linestyle="--", color="0.5", linewidth=1)
        ax.set_title(f"{nombre}   ROC-AUC macro {aucs[nombre]:.3f}")
        ax.set_xlabel("Tasa de falsos positivos")
        ax.set_ylabel("Tasa de verdaderos positivos")
        ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    ruta_roc = destino / "fig10_roc_por_clase.png"
    fig.savefig(ruta_roc, dpi=300, bbox_inches="tight")
    plt.close(fig)
    tabla = pd.concat(tablas, ignore_index=True)
    tabla.to_csv(destino / "tabla_tn_fp_fn_tp.csv", index=False)
    return aucs, ruta_cm, ruta_roc, tabla



if "df" in dir() and "TARGET_COL" in dir() and "DecisionTreeC45" in dir() and "RandomForestWrap" in dir():
    _X_fig, _y_fig = _separar_xy(df, TARGET_COL)
    _Xtr, _Xte, _ytr, _yte = train_test_split(
        _X_fig, _y_fig, test_size=0.2, random_state=SEED, stratify=_y_fig,
    )
    _dt_fig = DecisionTreeC45(random_state=SEED).fit(_Xtr, _ytr)
    _rf_fig = RandomForestWrap(
        n_estimators=100, max_features="sqrt", min_samples_leaf=1,
        bootstrap=True, random_state=SEED,
    ).fit(_Xtr, _ytr)
    orden_clases = list(_dt_fig.classes_)
    _aucs_fig, _ruta_cm, _ruta_roc, _tabla_tf = figura_confusion_roc(
        {"DecisionTreeC45": _dt_fig, "RandomForestWrap": _rf_fig},
        _Xte, _yte, orden_clases,
    )
    print(f"✓ Fig. 9  {_ruta_cm}")
    print(f"✓ Fig. 10 {_ruta_roc}")
    print(f"  clases (orden de las columnas de predict_proba): {orden_clases}")
    if len(orden_clases) == 2:
        print(f"  binario: negativa={orden_clases[0]}  positiva={orden_clases[1]}  (classes_[1], sale del dataset)")
    else:
        print(f"  {len(orden_clases)} clases: la matriz usa esos nombres; TN/FP/FN/TP van por clase en la tabla")
    display(_tabla_tf)
    for _nombre, _valor in _aucs_fig.items():
        print(f"  ROC-AUC macro {_nombre}: {_valor:.4f}")
else:
    print("ℹ️  Fig. 9: faltan df, DecisionTreeC45 o RandomForestWrap.")


✓ Fig. 9  figuras\fig9_matrices_confusion.png
✓ Fig. 10 figuras\fig10_roc_por_clase.png
  clases (orden de las columnas de predict_proba): [np.str_('ausencia'), np.str_('presencia')]
  binario: negativa=ausencia  positiva=presencia  (classes_[1], sale del dataset)


,modelo,clase_positiva,TP,FP,TN,FN
0,DecisionTreeC45,ausencia,57,14,88,25
1,DecisionTreeC45,presencia,88,25,57,14
2,RandomForestWrap,ausencia,67,10,92,15
3,RandomForestWrap,presencia,92,15,67,10


  ROC-AUC macro DecisionTreeC45: 0.8194
  ROC-AUC macro RandomForestWrap: 0.9219


---
# 5. Experimentos: entender el comportamiento

**Slots: 4º (números) · 5º (figuras / bitácora)** · Cards 36–44

**Qué es:** barridos controlados. No es “probar de todo”: cada experimento responde una pregunta.

| Experimento | Pregunta que responde |
|---|---|
| E2–E4 | ¿Cómo afectan profundidad / hojas mínimas / `min_gain_ratio` al DT? |
| E6–E7 | ¿Cuántos árboles y qué `max_features` necesita el RF? |
| E8 | ¿Cuántos datos necesita cada modelo? (learning curve) |
| E9 | ¿Hay overfitting? (brecha train/test) |
| E10–E11 | ¿Usan las mismas variables? ¿A qué costo de tamaño? |
| E13 | ¿En qué clases se confunden? |
| E12 (opc.) | ¿Qué pasa con ruido / faltantes artificiales? |

**Por qué:** el informe pide simulaciones y resultados, no solo “corrimos el default”.

**Trampa a evitar:** elegir hiperparámetros con un GridSearch y reportar *ese mismo* score. Está sesgado. Usar **holdout final** o **CV anidado** (card 43).

**Bitácora de roturas** (llenar mientras prueban):

| # | Caso | Qué pasó | Causa | Fix |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |

### Fig. 7 — Brecha train/test

La brecha es la media de accuracy en train menos la media en test, en las 25 particiones del `cv` compartido. Las barras comparan el árbol sin poda, el árbol con poda (`pruning_confidence=0.25`) y el bosque (100 árboles, `max_features="sqrt"`).

| | Train | Test | Brecha |
|---|---|---|---|
| DT sin poda | 0,9285 ± 0,0086 | 0,7850 ± 0,0265 | 0,1435 |
| DT con poda | 0,8663 ± 0,0151 | 0,7741 ± 0,0284 | 0,0922 |
| RF | 1,0000 ± 0,0000 | 0,8187 ± 0,0305 | 0,1813 |

Sin poda el árbol llega a 0,9285 en el train y la brecha es 0,1435. Con poda la brecha baja a 0,0922 y la accuracy de prueba también baja. El bosque acierta todo el train y la brecha queda en 0,1813, con una accuracy de prueba más alta que la del árbol. La figura queda en `figuras/fig7_brecha_train_test.png`.


In [24]:
# 5. E9 — Fig. 7, brecha train/test
# brecha = media(accuracy train) - media(accuracy test), sobre el cv compartido.


def construir_fig7(X, y):
    """Fig. 7: train y test del árbol con poda, sin poda, y del bosque."""
    if "cv" not in globals() or not isinstance(cv, RepeatedStratifiedKFold):
        raise RuntimeError("Falta el cv compartido. Correr la celda 1.1 antes.")
    import matplotlib.pyplot as plt

    variantes = (
        ("DT sin poda", DecisionTreeC45(pruning=False, random_state=SEED)),
        ("DT con poda", DecisionTreeC45(pruning=True, pruning_confidence=0.25, random_state=SEED)),
        ("RF", RandomForestWrap(
            n_estimators=100,
            max_depth=None,
            max_features="sqrt",
            min_samples_leaf=1,
            bootstrap=True,
            random_state=SEED,
        )),
    )
    filas = []
    for nombre, modelo in variantes:
        print(f"  E9 ajustando {nombre}", flush=True)
        resultado = cross_validate(
            modelo,
            X,
            y,
            cv=cv,
            scoring="accuracy",
            n_jobs=_trabajos_cv(),
            return_train_score=True,
            error_score="raise",
        )
        train = np.asarray(resultado["train_score"], dtype=float)
        test = np.asarray(resultado["test_score"], dtype=float)
        if len(train) != 25:
            raise RuntimeError(f"{nombre}: se esperaban 25 puntuaciones.")
        filas.append({
            "modelo": nombre,
            "train_media": float(train.mean()),
            "train_desvio": float(train.std(ddof=1)),
            "test_media": float(test.mean()),
            "test_desvio": float(test.std(ddof=1)),
            "brecha": float(train.mean() - test.mean()),
        })
    tabla = pd.DataFrame(filas)

    fig, ax = plt.subplots(figsize=(8, 4.6))
    x = np.arange(len(tabla))
    ancho = 0.36
    ax.bar(
        x - ancho / 2, tabla["train_media"], ancho,
        yerr=tabla["train_desvio"], capsize=3, label="Train",
    )
    ax.bar(
        x + ancho / 2, tabla["test_media"], ancho,
        yerr=tabla["test_desvio"], capsize=3, label="Test",
    )
    for i, fila in tabla.iterrows():
        ax.text(i, 1.02, f"brecha {fila['brecha']:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(list(tabla["modelo"]))
    ax.set_ylabel("Accuracy")
    ax.set_ylim(0, 1.14)
    ax.set_title("Fig. 7  Brecha train/test")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, frameon=False)
    fig.tight_layout()
    DIR_FIGURAS.mkdir(exist_ok=True)
    ruta = DIR_FIGURAS / "fig7_brecha_train_test.png"
    fig.savefig(ruta, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return tabla, ruta


if "df" in dir() and "TARGET_COL" in dir() and "DecisionTreeC45" in dir() and "RandomForestWrap" in dir():
    _X_e9, _y_e9 = _separar_xy(df, TARGET_COL)
    tabla_e9, _ruta_e9 = construir_fig7(_X_e9, _y_e9)
    print(f"✓ Fig. 7  {_ruta_e9}")
    display(tabla_e9.round(4))
else:
    print("ℹ️  Fig. 7: faltan df, DecisionTreeC45 o RandomForestWrap.")


  E9 ajustando DT sin poda


  E9 ajustando DT con poda


  E9 ajustando RF


✓ Fig. 7  figuras\fig7_brecha_train_test.png


,modelo,train_media,train_desvio,test_media,test_desvio,brecha
0,DT sin poda,0.9285,0.0086,0.7850,0.0265,0.1435
1,DT con poda,0.8663,0.0151,0.7741,0.0284,0.0922
2,RF,1.0000,0.0000,0.8187,0.0305,0.1813


### Holdout único (card 43, opción A)

Se elige la opción A. Un corte estratificado 80/20, `random_state=SEED`, se arma una sola vez sobre el dataset cargado.

`X_tr` es lo único que pueden ver los barridos (cards 36 y 37) y el `cv` compartido cuando se use para elegir hiperparámetros. `X_te` no entra en esos bucles. Se ajusta cada modelo en `X_tr` y se predice `X_te` una vez. El resultado queda en `figuras/holdout_final.json`. Si la celda se corre de nuevo con el mismo dataset, no vuelve a tocar `X_te`: lee el archivo.

Las tablas 2 y 5 y las figuras 7 y 9 se midieron antes de reservar este corte, sobre todas las filas. No son el holdout final.

Los números los imprime la celda sobre el dataset cargado. El corte de Breast Cancer (455 y 114, F1 macro 0,9246 y 0,9713) no se reutiliza.


In [25]:
# Opción A: los barridos usan X_tr. X_te se evalúa una vez.

import json


def partir_holdout(X, y):
    """Corte estratificado 80/20. La semilla es SEED, la misma de todo el trabajo."""
    return train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)


def evaluar_holdout_una_vez(modelos, X_tr, y_tr, X_te, y_te, ruta):
    """Ajusta en X_tr y predice X_te. Con la misma firma de datos, no reentrena."""
    firma = {
        "n_train": int(len(X_tr)),
        "n_test": int(len(X_te)),
        "clases": sorted(map(str, pd.unique(y_tr))),
        "modelos": list(modelos),
    }
    if ruta.exists():
        guardado = json.loads(ruta.read_text())
        if guardado.get("firma") == firma:
            print("Holdout ya evaluado para este dataset. X_te no se vuelve a tocar.")
            return pd.DataFrame(guardado["filas"])
    filas = []
    for nombre, modelo in modelos.items():
        modelo.fit(X_tr, y_tr)
        pred = modelo.predict(X_te)
        proba = modelo.predict_proba(X_te)
        fila = metricas(y_te, pred, proba, clases=list(modelo.classes_))
        fila["modelo"] = nombre
        filas.append(fila)
    tabla = pd.DataFrame(filas)
    ruta.parent.mkdir(exist_ok=True)
    ruta.write_text(json.dumps(
        {"firma": firma, "filas": tabla.to_dict(orient="records")},
        ensure_ascii=False, indent=2,
    ))
    return tabla


if "df" in dir() and "TARGET_COL" in dir() and "DecisionTreeC45" in dir() and "RandomForestWrap" in dir() and "metricas" in dir():
    _X_ho, _y_ho = _separar_xy(df, TARGET_COL)
    X_tr, X_te, y_tr, y_te = partir_holdout(_X_ho, _y_ho)
    if len(set(X_tr.index) & set(X_te.index)) != 0:
        raise RuntimeError("El holdout comparte filas con el entrenamiento.")
    _modelos_ho = {
        "DecisionTreeC45": DecisionTreeC45(random_state=SEED),
        "RandomForestWrap": RandomForestWrap(
            n_estimators=100, max_depth=None, max_features="sqrt",
            min_samples_leaf=1, bootstrap=True, random_state=SEED,
        ),
    }
    tabla_holdout = evaluar_holdout_una_vez(
        _modelos_ho, X_tr, y_tr, X_te, y_te, DIR_FIGURAS / "holdout_final.json",
    )
    print(f"Train {len(X_tr)}  |  holdout {len(X_te)}  (no entra en los barridos)")
    print("Los barridos de la celda siguiente tienen que usar X_tr, y_tr.")
    display(tabla_holdout.round(4))
else:
    print("ℹ️  Holdout: faltan df, los modelos o metricas. Correr las celdas anteriores.")

Train 736  |  holdout 184  (no entra en los barridos)
Los barridos de la celda siguiente tienen que usar X_tr, y_tr.


,accuracy,precision_macro,precision_weighted,recall_macro,recall_weighted,f1_macro,f1_weighted,roc_auc,modelo
0,0.7880,0.7908,0.7895,0.7789,0.7880,0.7819,0.7858,0.8192,DecisionTreeC45
1,0.8641,0.8650,0.8644,0.8595,0.8641,0.8616,0.8636,0.9219,RandomForestWrap


In [26]:
# ═══ Barridos E2–E11, solo sobre el entrenamiento ═══
# El holdout X_te no entra en estas grillas. Cada grilla mueve un parámetro.
# El elegido es el más chico cuya accuracy de test queda a menos de un desvío del máximo.

import json


def _media_metrica(tabla, metrica):
    sub = tabla[tabla["metrica"] == metrica]
    return {
        "train": float(sub["train"].mean()),
        "test": float(sub["test"].mean()),
        "desvio": float(sub["test"].std(ddof=1)),
        "brecha": float(sub["brecha"].mean()),
    }


def _elegir(filas):
    """filas tienen tamano (más grande = modelo más complejo) y accuracy de test."""
    mejor = max(filas, key=lambda fila: fila["accuracy_test"])
    piso = mejor["accuracy_test"] - mejor["accuracy_desvio"]
    banda = [fila for fila in filas if fila["accuracy_test"] >= piso - 1e-12]
    elegido = min(banda, key=lambda fila: fila["tamano"])
    orden = sorted(filas, key=lambda fila: fila["tamano"])
    sube = all(
        orden[i]["accuracy_test"] < orden[i + 1]["accuracy_test"] - 1e-12
        for i in range(len(orden) - 1)
    )
    return elegido, sube


def _fila_barrido(etiqueta, tamano, tabla, extra=None):
    acc = _media_metrica(tabla, "accuracy")
    f1 = _media_metrica(tabla, "f1_macro")
    fila = {
        "etiqueta": etiqueta,
        "tamano": tamano,
        "accuracy_train": acc["train"],
        "accuracy_test": acc["test"],
        "accuracy_desvio": acc["desvio"],
        "f1_test": f1["test"],
        "f1_desvio": f1["desvio"],
        "brecha": acc["brecha"],
    }
    if extra:
        fila.update(extra)
    return fila


def _contar_atributos(nodo, conteo):
    if getattr(nodo, "es_hoja", True) or not getattr(nodo, "hijos", None):
        return
    conteo[nodo.atributo] = conteo.get(nodo.atributo, 0) + 1
    for hijo in nodo.hijos.values():
        _contar_atributos(hijo, conteo)


def _nodos_arbol(modelo):
    raiz = modelo.arbol_.raiz
    conteo = {"nodos": 0}

    def visitar(nodo):
        conteo["nodos"] += 1
        for hijo in getattr(nodo, "hijos", {}).values():
            visitar(hijo)

    visitar(raiz)
    return conteo["nodos"]


if "X_tr" not in globals() or "y_tr" not in globals() or "X_te" not in globals():
    print("Falta el holdout. Corré la celda anterior: los barridos usan X_tr.")
else:
    DIR_FIGURAS.mkdir(exist_ok=True)
    print(f"Barridos sobre train {len(X_tr)}. Holdout {len(X_te)} no entra.")

    def _barrer_dt(nombre, valores, armar, tamano_de):
        filas = []
        for valor in valores:
            print(f"  {nombre} = {valor}", flush=True)
            tabla = evaluar_cv(armar(valor), X_tr, y_tr)
            ajustado = armar(valor).fit(X_tr, y_tr)
            filas.append(_fila_barrido(
                str(valor), tamano_de(valor), tabla,
                {"nodos": _nodos_arbol(ajustado)},
            ))
        return filas

    e2 = _barrer_dt(
        "max_depth",
        [1, 2, 3, 5, 8, None],
        lambda v: DecisionTreeC45(max_depth=v, random_state=SEED),
        lambda v: 10**9 if v is None else int(v),
    )
    for fila in e2:
        fila["etiqueta"] = "sin tope" if fila["tamano"] > 100 else fila["etiqueta"]
    e3 = _barrer_dt(
        "min_samples_leaf",
        [1, 2, 5, 10, 20, 50],
        lambda v: DecisionTreeC45(min_samples_leaf=v, random_state=SEED),
        lambda v: -int(v),
    )
    e4 = _barrer_dt(
        "min_gain_ratio",
        [0.0, 0.01, 0.05, 0.1, 0.2],
        lambda v: DecisionTreeC45(min_gain_ratio=v, random_state=SEED),
        lambda v: -float(v),
    )

    def _barrer_rf(nombre, valores, armar, tamano_de):
        filas = []
        for valor in valores:
            print(f"  {nombre} = {valor}", flush=True)
            tabla = evaluar_cv(armar(valor), X_tr, y_tr)
            filas.append(_fila_barrido(str(valor), tamano_de(valor), tabla))
        return filas

    e6 = _barrer_rf(
        "n_estimators",
        [1, 10, 50, 100, 200],
        lambda v: RandomForestWrap(n_estimators=v, random_state=SEED),
        lambda v: int(v),
    )
    e7 = _barrer_rf(
        "max_features",
        ["log2", "sqrt", None],
        lambda v: RandomForestWrap(max_features=v, random_state=SEED),
        lambda v: {None: 3, "sqrt": 2, "log2": 1}[v],
    )
    for fila, valor in zip(e7, ["log2", "sqrt", None]):
        fila["etiqueta"] = "todos" if valor is None else str(valor)

    fracciones = [0.2, 0.4, 0.6, 0.8, 1.0]
    e8 = []
    for frac in fracciones:
        if frac == 1:
            Xs, ys = X_tr, y_tr
        else:
            Xs, _, ys, _ = train_test_split(
                X_tr, y_tr, train_size=frac, random_state=SEED, stratify=y_tr,
            )
        for nombre, modelo in (
            ("DecisionTreeC45", DecisionTreeC45(random_state=SEED)),
            ("RandomForestWrap", RandomForestWrap(random_state=SEED)),
        ):
            print(f"  learning {nombre} {frac:.0%}", flush=True)
            tabla = evaluar_cv(modelo, Xs, ys)
            fila = _fila_barrido(nombre, frac, tabla, {"n": int(len(Xs))})
            e8.append(fila)

    dt_base = DecisionTreeC45(random_state=SEED).fit(X_tr, y_tr)
    uso = {}
    _contar_atributos(dt_base.arbol_.raiz, uso)
    rf_base = RandomForestWrap(random_state=SEED).fit(X_tr, y_tr)
    pre = rf_base.pipeline_.named_steps["preprocess"]
    rf = rf_base.pipeline_.named_steps["rf"]
    bruto = pd.Series(rf.feature_importances_, index=list(pre.get_feature_names_out()))
    por_columna = {}
    for clave, valor in bruto.items():
        origen = clave.split("__", 1)[-1]
        for col in sorted(X_tr.columns, key=len, reverse=True):
            if origen == col or origen.startswith(col + "_"):
                por_columna[col] = por_columna.get(col, 0.0) + float(valor)
                break
    e10 = pd.DataFrame({
        "atributo": list(X_tr.columns),
        "usos_dt": [int(uso.get(col, 0)) for col in X_tr.columns],
        "importancia_rf": [por_columna.get(col, 0.0) for col in X_tr.columns],
    }).sort_values("importancia_rf", ascending=False)

    def _guardar(filas, nombre):
        tabla = pd.DataFrame(filas)
        tabla.to_csv(DIR_FIGURAS / nombre, index=False)
        return tabla

    tablas = {
        "e2": _guardar(e2, "barrido_e2_profundidad.csv"),
        "e3": _guardar(e3, "barrido_e3_hoja.csv"),
        "e4": _guardar(e4, "barrido_e4_gain.csv"),
        "e6": _guardar(e6, "barrido_e6_arboles.csv"),
        "e7": _guardar(e7, "barrido_e7_max_features.csv"),
        "e8": _guardar(e8, "barrido_e8_learning.csv"),
    }
    e10.to_csv(DIR_FIGURAS / "barrido_e10_importancias.csv", index=False)
    tablas["e2"][["etiqueta", "nodos", "accuracy_test", "accuracy_desvio"]].to_csv(
        DIR_FIGURAS / "barrido_e11_tamano.csv", index=False,
    )

    elecciones = {}
    for clave, filas in (("e2", e2), ("e3", e3), ("e4", e4), ("e6", e6), ("e7", e7)):
        elegido, sube = _elegir(filas)
        elecciones[clave] = {
            "etiqueta": elegido["etiqueta"],
            "accuracy_test": elegido["accuracy_test"],
            "accuracy_desvio": elegido["accuracy_desvio"],
            "sube_sin_freno": bool(sube),
        }
        print(
            f"{clave}: elegido {elegido['etiqueta']} "
            f"(acc {elegido['accuracy_test']:.4f}). sube sin freno: {sube}"
        )

    def _valor_dt(etiqueta, crudo):
        if etiqueta == "sin tope":
            return None
        return crudo

    profundidad = _valor_dt(elecciones["e2"]["etiqueta"], elecciones["e2"]["etiqueta"])
    if profundidad not in (None, "sin tope"):
        profundidad = int(float(profundidad))
    elif profundidad == "sin tope":
        profundidad = None
    hoja = int(float(elecciones["e3"]["etiqueta"]))
    # e3 etiqueta es el valor de hoja; el tamano interno está negado y no se guarda
    gain = float(elecciones["e4"]["etiqueta"])
    n_arboles = int(float(elecciones["e6"]["etiqueta"]))
    feats = elecciones["e7"]["etiqueta"]
    max_features = None if feats == "todos" else feats

    modelos_elegidos = {
        "DecisionTreeC45_elegido": DecisionTreeC45(
            max_depth=profundidad,
            min_samples_leaf=hoja,
            min_gain_ratio=gain,
            random_state=SEED,
        ),
        "RandomForestWrap_elegido": RandomForestWrap(
            n_estimators=n_arboles,
            max_features=max_features,
            random_state=SEED,
        ),
    }
    tabla_elegida = evaluar_holdout_una_vez(
        modelos_elegidos, X_tr, y_tr, X_te, y_te, DIR_FIGURAS / "holdout_elegido.json",
    )
    # esa función cachea por firma de tamaños. Forzamos este archivo distinto.
    print("Holdout de los valores elegidos (no es el score con el que se eligieron):")
    display(tabla_elegida.round(4))

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.errorbar(
        range(len(e2)), [f["accuracy_test"] for f in e2],
        yerr=[f["accuracy_desvio"] for f in e2], marker="o", capsize=3,
    )
    ax.set_xticks(range(len(e2)))
    ax.set_xticklabels([f["etiqueta"] for f in e2])
    ax.set_xlabel("Profundidad")
    ax.set_ylabel("Accuracy de test")
    ax.set_title("Fig. 6a  Profundidad del árbol, solo entrenamiento")
    fig.tight_layout()
    fig.savefig(DIR_FIGURAS / "fig6a_profundidad.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    for nombre in ("DecisionTreeC45", "RandomForestWrap"):
        sub = [f for f in e8 if f["etiqueta"] == nombre]
        ax.errorbar(
            [f["n"] for f in sub], [f["accuracy_test"] for f in sub],
            yerr=[f["accuracy_desvio"] for f in sub], marker="o", capsize=3, label=nombre,
        )
    ax.set_xlabel("Filas de entrenamiento")
    ax.set_ylabel("Accuracy de test")
    ax.set_title("Fig. 8  Curva de aprendizaje")
    ax.legend()
    fig.tight_layout()
    fig.savefig(DIR_FIGURAS / "fig8_learning_curve.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    (DIR_FIGURAS / "barridos_eleccion.json").write_text(json.dumps({
        "elecciones": elecciones,
        "dt": {"max_depth": profundidad, "min_samples_leaf": hoja, "min_gain_ratio": gain},
        "rf": {"n_estimators": n_arboles, "max_features": feats},
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    display(e10.round(4))


Barridos sobre train 736. Holdout 184 no entra.
  max_depth = 1


  max_depth = 2


  max_depth = 3


  max_depth = 5


  max_depth = 8


  max_depth = None


... (29 outputs left out)

e2: elegido 2 (acc 0.7286). sube sin freno: True
e3: elegido 50 (acc 0.7653). sube sin freno: False
e4: elegido 0.1 (acc 0.7432). sube sin freno: True
e6: elegido 10 (acc 0.7859). sube sin freno: True
e7: elegido log2 (acc 0.8093). sube sin freno: False


Holdout de los valores elegidos (no es el score con el que se eligieron):


,accuracy,precision_macro,precision_weighted,recall_macro,recall_weighted,f1_macro,f1_weighted,roc_auc,modelo
0,0.7935,0.7924,0.7977,0.7958,0.7935,0.7926,0.7941,0.8057,DecisionTreeC45_elegido
1,0.8370,0.8377,0.8372,0.8314,0.8370,0.8336,0.8362,0.8923,RandomForestWrap_elegido


,atributo,usos_dt,importancia_rf
2,cp,1,0.1502
4,chol,24,0.1316
8,exang,35,0.1162
0,age,1,0.1126
7,thalach,9,0.1061
9,oldpeak,5,0.0861
3,trestbps,25,0.0734
1,sex,11,0.0499
12,thal,16,0.0445
6,restecg,20,0.0424


In [27]:
# ═══ Figuras y tablas ya medidas (tarjetas 42 y 44) ═══
# No reentrena. No pisa un archivo que esta corrida ya escribió.

import shutil
from pathlib import Path

_origen_figuras = Path("bitacoras/nadine/figuras/figuras")
_destino_figuras = DIR_FIGURAS if "DIR_FIGURAS" in globals() else Path("figuras")
_destino_figuras.mkdir(parents=True, exist_ok=True)
_archivos_entrega = [
    "fig4_dt_vs_rf.png",
    "fig7_brecha_train_test.png",
    "fig9_matrices_confusion.png",
    "fig10_roc_por_clase.png",
    "tabla2_dt_vs_rf.tex",
    "tabla5_poda.tex",
    "tabla_tn_fp_fn_tp.csv",
    "holdout_final.json",
]
if not _origen_figuras.exists():
    print(f"No está {_origen_figuras}. No se copió nada.")
else:
    for _nombre in _archivos_entrega:
        _src = _origen_figuras / _nombre
        _dst = _destino_figuras / _nombre
        if _dst.exists():
            print(f"se deja {_dst}")
            continue
        # El archivo de Nadine es Breast Cancer. No rellenar un hueco de esta corrida.
        if "breast" not in str(globals().get("NOMBRE_DATASET", "")).lower():
            print(f"no se copia {_nombre}: el dataset cargado no es el respaldo")
            continue
        if not _src.exists():
            print(f"Falta {_nombre}")
            continue
        shutil.copy2(_src, _dst)
        print(f"copiado {_dst}")


se deja figuras\fig4_dt_vs_rf.png
se deja figuras\fig7_brecha_train_test.png
se deja figuras\fig9_matrices_confusion.png
se deja figuras\fig10_roc_por_clase.png
se deja figuras\tabla2_dt_vs_rf.tex
se deja figuras\tabla5_poda.tex
se deja figuras\tabla_tn_fp_fn_tp.csv
se deja figuras\holdout_final.json


---
# 6. Interfaz e informe

### 6.1 UI en Colab

**Slot: 5º** · Tarjetas 45–49

El panel elige dataset, clase y atributos, los hiperparámetros del árbol y del bosque, y el corte de prueba. **Ejecutar** compara los dos modelos. **Bajar CSV** guarda media, desvío y el holdout.

El menú lista los CSV que haya en `data/`. La clase inicial es la columna `target` si existe y, si no, la última. El tipo de cada atributo sale de los valores: texto y bool son categorías, un entero con hasta 10 valores distintos es un código, y el resto numérico es continuo. Las tablas del informe siguen siendo las de Heart Disease. Esta celda no las vuelve a calcular.

El `cv` de 25 particiones del informe no se toca. Acá el tope es 10 particiones, porque el árbol propio tarda. Con los valores iniciales (2 folds, 1 repetición) la prueba es la corta.

Si los controles no se dibujan, el resultado igual se imprime en la celda.

In [ ]:
# ═══ 6.1 Panel (tarjetas 45–49) ═══
# ipywidgets 7.7.1, que es la versión de Colab.
# Esta celda solo arma los controles. El entrenamiento arranca con Ejecutar.

import traceback
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Tope del panel, no del informe. El árbol propio anda por los 20 s por ajuste.
_TOPE_PARTICIONES = 10
_METRICAS_PANEL = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted", "roc_auc"]
_panel = {"tabla": None, "cargando": False}


class _Aviso(Exception):
    """Corte esperado: falta un dato o el tope de particiones."""


def _st():
    return {"description_width": "140px"}


def _ly(ancho="420px"):
    return widgets.Layout(width=ancho)


def _profundidad(valor):
    # El slider no acepta None. -1 en el botón es "sin límite".
    return None if int(valor) < 0 else int(valor)


def _max_features(valor):
    if valor == "todos":
        return None
    if valor == "0.3":
        return 0.3
    return valor


def _frame_dataset(clave):
    if clave == "respaldo":
        frame, nombre = _cargar_respaldo(), "breast_cancer (respaldo)"
    else:
        frame = _leer_csv(clave)
        frame.columns = frame.columns.str.strip()
        nombre = clave
    if "preprocess" in globals():
        frame = preprocess(frame, _elegir_clase(frame.columns))
    return frame, nombre


def _nuevo_dt():
    return DecisionTreeC45(
        max_depth=_profundidad(_dt_depth.value),
        min_samples_split=int(_dt_split.value),
        min_samples_leaf=int(_dt_leaf.value),
        min_gain_ratio=float(_dt_gain.value),
        pruning=bool(_dt_poda.value),
        pruning_confidence=float(_dt_cf.value),
        random_state=SEED,
    )


def _nuevo_rf():
    return RandomForestWrap(
        n_estimators=int(_rf_arboles.value),
        max_depth=_profundidad(_rf_depth.value),
        max_features=_max_features(_rf_feats.value),
        min_samples_leaf=int(_rf_leaf.value),
        bootstrap=True,
        random_state=SEED,
    )


def _puntajes(modelo, y_real, pred, proba):
    puntos = metricas(y_real, pred, proba, clases=list(modelo.classes_))
    # En multiclase la clave viene como roc_auc_macro. La tabla usa un solo nombre.
    if "roc_auc" not in puntos and "roc_auc_macro" in puntos:
        puntos["roc_auc"] = puntos["roc_auc_macro"]
    return puntos


def _poner_features(frame, target):
    columnas = [c for c in frame.columns if c != target]
    # Vaciar antes de cambiar las opciones, si no el valor viejo revienta.
    _features.value = ()
    _features.options = columnas
    _features.value = tuple(columnas)


def _sincronizar_columnas():
    _panel["cargando"] = True
    try:
        frame, _nombre = _frame_dataset(_dd_dataset.value)
        columnas = list(frame.columns)
        preferido = _elegir_clase(columnas)
        _dd_target.options = columnas
        _dd_target.value = preferido
        _poner_features(frame, preferido)
    finally:
        _panel["cargando"] = False


def _al_cambiar_dataset(cambio):
    if cambio["name"] != "value" or _panel["cargando"]:
        return
    _sincronizar_columnas()


def _al_cambiar_target(cambio):
    if cambio["name"] != "value" or _panel["cargando"]:
        return
    frame, _nombre = _frame_dataset(_dd_dataset.value)
    _poner_features(frame, cambio["new"])


def _resumen_cv(registros):
    tabla = (
        pd.DataFrame(registros)
        .groupby(["modelo", "metrica"])["valor"]
        .agg(["mean", "std"])
        .reset_index()
    )
    tabla.columns = ["modelo", "metrica", "media", "desvio"]
    tabla["desvio"] = tabla["desvio"].fillna(0.0)
    orden = {nombre: i for i, nombre in enumerate(_METRICAS_PANEL)}
    tabla["_orden"] = tabla["metrica"].map(orden)
    return tabla.sort_values(["modelo", "_orden"]).drop(columns="_orden").reset_index(drop=True)


def _figuras(modelos, X_te, y_te, tabla):
    carpeta = DIR_FIGURAS if "DIR_FIGURAS" in globals() else Path("figuras")
    carpeta.mkdir(exist_ok=True)

    fig, ejes = plt.subplots(1, 2, figsize=(9.5, 4))
    for eje, (nombre, modelo) in zip(ejes, modelos):
        pred = modelo.predict(X_te)
        matriz = confusion_matrix(y_te, pred, labels=list(modelo.classes_))
        ConfusionMatrixDisplay(matriz, display_labels=list(modelo.classes_)).plot(
            ax=eje, colorbar=False,
        )
        eje.set_title(nombre)
    fig.tight_layout()
    ruta_cm = carpeta / "panel_confusion.png"
    fig.savefig(ruta_cm, dpi=120, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(ruta_cm)))

    querer = ["accuracy", "f1_macro", "recall_macro"]
    fig, eje = plt.subplots(figsize=(7, 3.6))
    x = np.arange(len(querer))
    ancho = 0.36
    for despl, (nombre, _modelo) in zip((-ancho / 2, ancho / 2), modelos):
        altos = []
        for metrica in querer:
            fila = tabla[(tabla["modelo"] == nombre) & (tabla["metrica"] == metrica)]
            altos.append(float(fila["holdout"].iloc[0]) if len(fila) else 0.0)
        eje.bar(x + despl, altos, ancho, label=nombre)
    eje.set_xticks(x)
    eje.set_xticklabels(querer)
    eje.set_ylim(0, 1.05)
    eje.set_ylabel("Holdout")
    eje.legend()
    fig.tight_layout()
    ruta_barras = carpeta / "panel_barras.png"
    fig.savefig(ruta_barras, dpi=120, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(ruta_barras)))
    return ruta_cm, ruta_barras


def _evaluar():
    features = list(_features.value)
    target = _dd_target.value
    if not features:
        raise _Aviso("Elegí al menos un atributo.")
    if target in features:
        raise _Aviso("La clase no puede estar también entre los atributos.")

    n_splits = int(_cv_splits.value)
    n_repeats = int(_cv_repeats.value)
    particiones = n_splits * n_repeats
    if particiones > _TOPE_PARTICIONES:
        raise _Aviso(
            f"{n_splits} × {n_repeats} = {particiones} particiones. "
            f"El tope del panel es {_TOPE_PARTICIONES}: el árbol propio tarda unos 20 s por ajuste."
        )

    frame, nombre = _frame_dataset(_dd_dataset.value)
    if target not in frame.columns:
        raise _Aviso(f"No está la columna {target}.")
    datos = frame.loc[:, features + [target]].dropna(subset=[target]).reset_index(drop=True)
    if datos[target].nunique() < 2:
        raise _Aviso("Hace falta más de una clase.")
    if int(datos[target].value_counts().min()) < n_splits:
        raise _Aviso("La clase más chica no alcanza para tantos folds.")

    X = datos.drop(columns=[target])
    y = datos[target]
    test_size = float(_cv_test.value)
    try:
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=test_size, random_state=SEED, stratify=y,
        )
    except ValueError as exc:
        raise _Aviso(f"No se pudo partir el holdout: {exc}") from exc

    X_tr = X_tr.reset_index(drop=True)
    X_te = X_te.reset_index(drop=True)
    y_tr = y_tr.reset_index(drop=True)
    y_te = y_te.reset_index(drop=True)
    if int(y_tr.value_counts().min()) < n_splits:
        raise _Aviso("En el entrenamiento la clase más chica no entra en esos folds. Subí el tamaño de prueba o bajá los folds.")

    # Corte local. El cv de 25 particiones del setup queda como está.
    cortes = RepeatedStratifiedKFold(
        n_splits=n_splits, n_repeats=n_repeats, random_state=SEED,
    )
    fabricas = (("DecisionTreeC45", _nuevo_dt), ("RandomForestWrap", _nuevo_rf))
    print(
        f"{nombre}: {len(datos)} filas, holdout {len(X_te)}, "
        f"{particiones} particiones solo sobre las {len(X_tr)} de entrenamiento."
    )
    print("El árbol propio es el lento. El bosque de sklearn va rápido.")

    registros = []
    pasos = list(cortes.split(X_tr, y_tr))
    for i, (tr_idx, va_idx) in enumerate(pasos, start=1):
        print(f"Partición {i}/{len(pasos)}")
        for nombre_modelo, fabrica in fabricas:
            modelo = fabrica()
            modelo.fit(X_tr.iloc[tr_idx], y_tr.iloc[tr_idx])
            pred = modelo.predict(X_tr.iloc[va_idx])
            proba = modelo.predict_proba(X_tr.iloc[va_idx])
            puntos = _puntajes(modelo, y_tr.iloc[va_idx], pred, proba)
            for metrica in _METRICAS_PANEL:
                if metrica in puntos:
                    registros.append({
                        "modelo": nombre_modelo,
                        "metrica": metrica,
                        "valor": puntos[metrica],
                    })

    print("Ajuste final, una sola vez, para el holdout.")
    entrenados = []
    holdout = []
    for nombre_modelo, fabrica in fabricas:
        modelo = fabrica()
        modelo.fit(X_tr, y_tr)
        pred = modelo.predict(X_te)
        proba = modelo.predict_proba(X_te)
        puntos = _puntajes(modelo, y_te, pred, proba)
        entrenados.append((nombre_modelo, modelo))
        for metrica in _METRICAS_PANEL:
            if metrica in puntos:
                holdout.append({
                    "modelo": nombre_modelo,
                    "metrica": metrica,
                    "holdout": puntos[metrica],
                })

    tabla = _resumen_cv(registros).merge(
        pd.DataFrame(holdout), on=["modelo", "metrica"], how="left",
    )
    _panel["tabla"] = tabla
    display(tabla.round(4))
    ruta_cm, ruta_barras = _figuras(entrenados, X_te, y_te, tabla)

    carpeta = ruta_cm.parent
    ruta_csv = carpeta / "panel_resultados.csv"
    tabla.to_csv(ruta_csv, index=False)
    print(f"CSV listo en {ruta_csv}. Matrices: {ruta_cm.name}, barras: {ruta_barras.name}.")
    return tabla.round(4).to_string(index=False)


def _correr(boton):
    boton.disabled = True
    texto = ""
    try:
        with _salida:
            _salida.clear_output(wait=True)
            try:
                texto = _evaluar()
            except _Aviso as aviso:
                texto = str(aviso)
                print(texto)
    except Exception:
        texto = traceback.format_exc()
        with _salida:
            print(texto)
    finally:
        boton.disabled = False
    # Fuera del Output: si el widget no se ve, el resultado igual queda en la celda.
    if texto:
        print(texto)


def _descargar(_boton):
    tabla = _panel.get("tabla")
    if tabla is None:
        print("Todavía no hay resultados. Apretá Ejecutar.")
        return
    carpeta = DIR_FIGURAS if "DIR_FIGURAS" in globals() else Path("figuras")
    carpeta.mkdir(exist_ok=True)
    ruta = carpeta / "panel_resultados.csv"
    tabla.to_csv(ruta, index=False)
    print(f"CSV: {ruta}")
    try:
        from google.colab import files
        files.download(str(ruta))
    except ImportError:
        print("Fuera de Colab no se abre la descarga. El archivo quedó en el disco.")


_faltan = [
    nombre for nombre in (
        "DecisionTreeC45", "RandomForestWrap", "metricas", "SEED",
        "_cargar_respaldo", "_leer_csv", "TARGET_COL",
    )
    if nombre not in globals()
]

print("Panel de las tarjetas 45–49. Si los controles no se dibujan, Ejecutar igual imprime en la celda.")
print(f"ipywidgets {widgets.__version__}.")

if _faltan:
    print("Faltan celdas anteriores:", ", ".join(_faltan))
else:
    _opciones = []
    if Path("data").exists():
        for _ruta in sorted(Path("data").rglob("*.csv")):
            _texto = _ruta.as_posix()
            _opciones.append((_texto, _texto))
    if not _opciones:
        print("No hay CSV en data/. Queda el respaldo.")
        _opciones = [("Breast Cancer (respaldo)", "respaldo")]
    _valor_dataset = RUTA_DATOS if any(ruta == RUTA_DATOS for _etiq, ruta in _opciones) else _opciones[0][1]
    _dd_dataset = widgets.Dropdown(options=_opciones, value=_valor_dataset, description="Dataset", style=_st(), layout=_ly())
    _dd_target = widgets.Dropdown(description="Clase", style=_st(), layout=_ly())
    _features = widgets.SelectMultiple(description="Atributos", rows=8, style=_st(), layout=_ly())

    _dt_depth = widgets.ToggleButtons(
        options=[("Sin límite", -1), ("1", 1), ("2", 2), ("3", 3), ("5", 5), ("8", 8)],
        value=-1, description="Profundidad", style=_st(),
    )
    _dt_split = widgets.IntSlider(value=2, min=2, max=40, description="Mín. para partir", style=_st(), layout=_ly())
    _dt_leaf = widgets.IntSlider(value=1, min=1, max=30, description="Mín. por hoja", style=_st(), layout=_ly())
    _dt_gain = widgets.FloatSlider(value=0.0, min=0.0, max=0.2, step=0.01, description="Gain ratio mín.", style=_st(), layout=_ly(), readout_format=".2f")
    _dt_poda = widgets.Checkbox(value=False, description="Poda pesimista")
    _dt_cf = widgets.FloatSlider(value=0.25, min=0.05, max=0.5, step=0.05, description="Confianza (CF)", style=_st(), layout=_ly(), readout_format=".2f")

    _rf_arboles = widgets.IntSlider(value=100, min=1, max=200, description="Árboles", style=_st(), layout=_ly())
    _rf_depth = widgets.ToggleButtons(
        options=[("Sin límite", -1), ("2", 2), ("4", 4), ("8", 8)],
        value=-1, description="Profundidad", style=_st(),
    )
    _rf_feats = widgets.Dropdown(
        options=[("sqrt", "sqrt"), ("log2", "log2"), ("30%", "0.3"), ("todos", "todos")],
        value="sqrt", description="Atributos por corte", style=_st(), layout=_ly(),
    )
    _rf_leaf = widgets.IntSlider(value=1, min=1, max=20, description="Mín. por hoja", style=_st(), layout=_ly())

    _cv_splits = widgets.IntSlider(value=2, min=2, max=10, description="Folds", style=_st(), layout=_ly())
    _cv_repeats = widgets.IntSlider(value=1, min=1, max=10, description="Repeticiones", style=_st(), layout=_ly())
    _cv_test = widgets.FloatSlider(value=0.2, min=0.1, max=0.5, step=0.05, description="Tamaño de prueba", style=_st(), layout=_ly(), readout_format=".2f")

    _sincronizar_columnas()
    _dd_dataset.observe(_al_cambiar_dataset, names="value")
    _dd_target.observe(_al_cambiar_target, names="value")

    _caja_datos = widgets.VBox([
        widgets.HTML("Al cambiar el dataset se vuelve a armar la lista de la clase y de los atributos."),
        _dd_dataset, _dd_target, _features,
    ])
    _caja_dt = widgets.VBox([
        widgets.HTML("Profundidad «Sin límite» no manda un tope. La semilla es la del setup."),
        _dt_depth, _dt_split, _dt_leaf, _dt_gain, _dt_poda, _dt_cf,
    ])
    _caja_rf = widgets.VBox([
        widgets.HTML("«Todos» en atributos por corte apaga el submuestreo: queda bagging."),
        _rf_arboles, _rf_depth, _rf_feats, _rf_leaf,
    ])
    _caja_cv = widgets.VBox([
        widgets.HTML(
            "El tamaño de prueba es un holdout, una sola vez. "
            f"Los folds usan solo el entrenamiento. Tope: {_TOPE_PARTICIONES} particiones."
        ),
        _cv_splits, _cv_repeats, _cv_test,
    ])
    _tabs = widgets.Tab(children=[_caja_datos, _caja_dt, _caja_rf, _caja_cv])
    for _i, _titulo in enumerate(("Datos", "Árbol", "Bosque", "Validación")):
        _tabs.set_title(_i, _titulo)

    _boton = widgets.Button(description="Ejecutar", button_style="primary")
    _boton_csv = widgets.Button(description="Bajar CSV")
    _salida = widgets.Output()
    _boton.on_click(_correr)
    _boton_csv.on_click(_descargar)

    print("Valores iniciales: 2 folds y 1 repetición. Una segunda corrida borra la salida anterior.")
    try:
        display(widgets.VBox([_tabs, widgets.HBox([_boton, _boton_csv]), _salida]))
    except Exception as exc:
        print(f"El panel no se pudo mostrar: {type(exc).__name__}: {exc}")


### 6.2 Demo de punta a punta

Muestra el holdout que ya se midió (`figuras/holdout_final.json`). No vuelve a entrenar: un Run all no tiene que pagar otra vez las 25 particiones. Para cambiar parámetros se usa el panel de la celda 6.1.


In [29]:
# ═══ 6.2 Demo ═══
# No entrena. Muestra el holdout ya medido. Otro corte sale del panel 6.1.

import json
from pathlib import Path

_carpeta_demo = DIR_FIGURAS if "DIR_FIGURAS" in globals() else Path("figuras")
_ruta_holdout = _carpeta_demo / "holdout_final.json"
if not _ruta_holdout.exists():
    _ruta_holdout = Path("bitacoras/nadine/figuras/figuras/holdout_final.json")

if not _ruta_holdout.exists():
    print("No hay holdout_final.json. En el panel de la celda 6.1 apretá Ejecutar.")
else:
    _holdout = json.loads(_ruta_holdout.read_text(encoding="utf-8"))
    _firma = _holdout.get("firma", {})
    print(
        f"Holdout ya medido: train {_firma.get('n_train')} | "
        f"test {_firma.get('n_test')} | clases {_firma.get('clases')}"
    )
    display(pd.DataFrame(_holdout["filas"]).round(4))
    print("Esta celda no reentrena. Para otra corrida, usá Ejecutar en el panel.")


Holdout ya medido: train 736 | test 184 | clases ['ausencia', 'presencia']


,accuracy,precision_macro,precision_weighted,recall_macro,recall_weighted,f1_macro,f1_weighted,roc_auc,modelo
0,0.7880,0.7908,0.7895,0.7789,0.7880,0.7819,0.7858,0.8192,DecisionTreeC45
1,0.8641,0.8650,0.8644,0.8595,0.8641,0.8616,0.8636,0.9219,RandomForestWrap


Esta celda no reentrena. Para otra corrida, usá Ejecutar en el panel.


### 6.3 Informe LNCS

**Slot: 5º coordina · todos escriben su parte** · Cards 50–57

**Qué es:** el documento formal (intro, algoritmos, solución/problemas, resultados, conclusiones, referencias).

**Por qué:** pesa en la nota; plagio o referencia que no coincide con lo implementado = desaprobado.

**Hacer:** usar `informe/informe.tex`. Meter figuras del §5. La bitácora de roturas alimenta “problemas encontrados”. Cada uno redacta lo que construyó; el 5º unifica estilo y recorta a ≤ 12 carillas.

---
# 7. Cierre y entrega

**Slots: todos · 5º coordina** · Cards 58–62

**Qué es:** buffer para pulir, no para empezar cosas nuevas.

**Hacer:**
- `Restart and Run all` en Colab limpio
- Informe ≤ 12 carillas + ortografía
- Trello en orden
- Entregar **antes del 11/11/2026 23:59**

**Listo cuando:**
- [ ] DT propio funciona (sin usar `sklearn.tree` como modelo entregable)
- [ ] RF sklearn corre con la misma evaluación
- [ ] Hay comparación con métricas y al menos E1
- [ ] UI usable (o fallback impreso documentado)
- [ ] Informe + referencias verificadas
- [ ] Cada uno puede explicar su parte en el coloquio